# NLP-04 — 09_fairness_audit

Publication copy of the research notebook.

Saved outputs and unnecessary metadata have been removed. Private annotations, raw datasets and model checkpoints are retained separately.

Some cells require external research artifacts. The public notebook is not a standalone reproduction package.

In [ ]:
# Cell 1: Fairness audit setup and verification

from google.colab import drive
from pathlib import Path
import json
import hashlib
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.metrics import average_precision_score

drive.mount('/content/drive')

PROJECT_ROOT = Path(
    '/content/drive/MyDrive/NLP04_Toxic_Comment_Classification'
)

assert PROJECT_ROOT.is_dir()

CONFIG_DIR = PROJECT_ROOT / '00_config'
SRC_DIR = PROJECT_ROOT / '03_src'
DATA_DIR = PROJECT_ROOT / '01_data'
PROCESSED_DIR = DATA_DIR / 'processed'
MODELS_DIR = PROJECT_ROOT / '04_models'

NOTEBOOK_DIR = PROJECT_ROOT / '02_notebooks/09_fairness_audit'
RESULTS_DIR = NOTEBOOK_DIR / 'results'

TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR = RESULTS_DIR / 'figures'
LOGS_DIR = RESULTS_DIR / 'logs'
RUN_CONFIGS_DIR = RESULTS_DIR / 'configs'
PREDICTIONS_DIR = RESULTS_DIR / 'predictions'
PRIVATE_DIR = RESULTS_DIR / 'private_audit'

for path in [
    TABLES_DIR, FIGURES_DIR, LOGS_DIR,
    RUN_CONFIGS_DIR, PREDICTIONS_DIR, PRIVATE_DIR
]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 42
rng = np.random.default_rng(SEED)

# File utility functions

def read_json(path):
    return json.loads(Path(path).read_text())


def calculate_sha256(path):

    h = hashlib.sha256()

    with open(path, 'rb') as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b''
        ):
            h.update(chunk)

    return h.hexdigest()


def save_json_atomic(path, content):

    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')

    tmp.write_text(
        json.dumps(content, indent=4, allow_nan=False)
    )

    assert read_json(tmp) == content

    tmp.replace(path)


def save_csv_atomic(path, frame):

    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')

    frame.to_csv(tmp, index=False)

    tmp.replace(path)


def verify_file(record):

    path = (PROJECT_ROOT / record['path']).resolve()

    assert path.is_relative_to(PROJECT_ROOT.resolve())
    assert path.is_file(), path
    assert path.stat().st_size == record['size_bytes'], path
    assert calculate_sha256(path) == record['sha256'], path

    return path


# Restore the frozen evaluation protocol

REGISTRY_PATH = CONFIG_DIR / 'experiment_registry.csv'
DECISION_LOG_PATH = CONFIG_DIR / 'project_decisions.csv'

FROZEN_MANIFEST_PATH = (
    PROJECT_ROOT /
    '02_notebooks/06_model_optimization/results/configs/'
    'frozen_manifest.json'
)

EXPECTED_FROZEN_HASH = (
    '229adb5d6166a626f6b7f7fddecf38b4'
    'de58d21e56ec4367465ad29174ef2fd1'
)

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

frozen_manifest = read_json(FROZEN_MANIFEST_PATH)

assert frozen_manifest['thresholds_frozen']
assert frozen_manifest['model_parameters_frozen']

assert not frozen_manifest[
    'official_test_used_for_selection'
]

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

LABELS = frozen_manifest['labels']

assert LABELS == [
    'toxic', 'severe_toxic', 'obscene',
    'threat', 'insult', 'identity_hate'
]

FROZEN_MODELS = {
    entry['run_id']: entry
    for entry in frozen_manifest['models']
}

assert len(FROZEN_MODELS) == 19

# Restore the three selected audit models

selected = frozen_manifest['selected_models']

PRIMARY_RUN_ID = selected['primary_model']
BASELINE_RUN_ID = selected['best_classical']
DISTILBERT_RUN_ID = selected['lightweight_transformer']

assert PRIMARY_RUN_ID == 'S02_ROBERTA_FOCAL_FULL_S42'
assert BASELINE_RUN_ID == 'S01_LR_COMBINED_BALANCED_FULL_S42'
assert DISTILBERT_RUN_ID == 'T03_FOCAL_S42'

AUDIT_MODELS = {
    BASELINE_RUN_ID: 'Logistic Regression',
    DISTILBERT_RUN_ID: 'DistilBERT',
    PRIMARY_RUN_ID: 'RoBERTa'
}

AUDIT_RUN_IDS = list(AUDIT_MODELS)

# Verify completed previous notebooks

EVALUATION_RESULTS_DIR = (
    PROJECT_ROOT /
    '02_notebooks/07_final_evaluation/results'
)

EVALUATION_COMPLETION_PATH = (
    EVALUATION_RESULTS_DIR /
    'logs/evaluation_completion_summary.json'
)

ERROR_ANALYSIS_COMPLETION_PATH = (
    PROJECT_ROOT /
    '02_notebooks/08_error_analysis/results/logs/'
    'error_analysis_completion_summary.json'
)

evaluation_completion = read_json(
    EVALUATION_COMPLETION_PATH
)

error_completion = read_json(
    ERROR_ANALYSIS_COMPLETION_PATH
)

assert evaluation_completion['status'] == 'Completed'
assert evaluation_completion['evaluated_configurations'] == 19

assert error_completion['ai_assisted_review_completed']
assert error_completion['review_cases'] == 180

for record in [evaluation_completion, error_completion]:

    assert record['frozen_manifest_sha256'] == (
        EXPECTED_FROZEN_HASH
    )

    assert not record['thresholds_changed']
    assert not record['test_driven_selection_performed']
    assert not record['model_training_performed']

print('Previous notebooks and frozen protocol: Verified')

# Restore official test dataset

TEST_PATH = PROJECT_ROOT / evaluation_completion[
    'test_dataset_path'
]

EXPECTED_TEST_HASH = (
    'cddc70d876b48b9fe3d5db8109a7404'
    '30a7f4d9f50f9fcd77f4c7d566035f328'
)

assert calculate_sha256(
    TEST_PATH
) == EXPECTED_TEST_HASH

test_df = pd.read_parquet(
    TEST_PATH,
    columns=['id', 'comment_text', *LABELS]
)

assert len(test_df) == 63978
assert test_df['id'].is_unique
assert test_df['id'].notna().all()
assert test_df['comment_text'].notna().all()

y_test = test_df[LABELS].to_numpy(dtype=np.int8)
test_ids = test_df['id'].astype(str).to_numpy()

assert np.isin(y_test, [0, 1]).all()

# Restore normalized text groups for uncertainty analysis

inventory = {
    record['path']: record
    for record in evaluation_completion['artifact_inventory']
}

assert len(inventory) == 79


def verify_evaluation_artifact(path):

    key = str(Path(path).relative_to(PROJECT_ROOT))

    return verify_file(inventory[key])


normalized_text = (
    test_df['comment_text']
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

test_group_ids, unique_groups = pd.factorize(
    normalized_text,
    sort=False
)

assert len(unique_groups) == 63939

del normalized_text, unique_groups
gc.collect()

# Restore verified prediction index

index_path = PROJECT_ROOT / evaluation_completion[
    'prediction_index_path'
]

verify_evaluation_artifact(index_path)

assert calculate_sha256(index_path) == (
    evaluation_completion['prediction_index_sha256']
)

prediction_index = pd.read_csv(
    index_path
).set_index('Run ID')

assert prediction_index.index.is_unique
assert set(prediction_index.index) == set(FROZEN_MODELS)

per_label_path = verify_evaluation_artifact(
    EVALUATION_RESULTS_DIR /
    'tables/test_per_label_metrics.csv'
)

per_label_test = pd.read_csv(per_label_path)

assert len(per_label_test) == 228

# Verify existing predictions for all three audit models

audit_predictions = {}
audit_scores = {}
audit_binary_predictions = {}
verified_sources = []
setup_rows = []

for rid in AUDIT_RUN_IDS:

    record = prediction_index.loc[rid]

    prediction_path = PROJECT_ROOT / record[
        'Prediction Path'
    ]

    verify_evaluation_artifact(prediction_path)

    assert calculate_sha256(prediction_path) == (
        record['SHA256']
    )

    frame = pd.read_parquet(prediction_path)

    assert len(frame) == len(test_df)
    assert frame['id'].is_unique
    assert frame['run_id'].eq(rid).all()
    assert frame['split'].eq('test_official').all()

    np.testing.assert_array_equal(
        frame['id'].astype(str).to_numpy(),
        test_ids
    )

    np.testing.assert_array_equal(
        frame[[f'true_{l}' for l in LABELS]].to_numpy(),
        y_test
    )

    scores = frame[
        [f'score_{l}' for l in LABELS]
    ].to_numpy(dtype=float)

    entry = FROZEN_MODELS[rid]

    assert record['Score Type'] == entry['score_type']
    assert entry['comparison_operator'] == '>='

    assert np.isfinite(scores).all()
    assert ((scores >= 0) & (scores <= 1)).all()

    thresholds = np.array([
        entry['optimized_thresholds'][label]
        for label in LABELS
    ])

    pred = scores >= thresholds

    np.testing.assert_array_equal(
        frame[
            [f'pred_optimized_{l}' for l in LABELS]
        ].to_numpy(),
        pred
    )

    reference = per_label_test[
        (per_label_test['Run ID'] == rid) &
        (per_label_test['Operating Point'] == 'optimized')
    ].set_index('Label').loc[LABELS]

    # Verify confusion counts and ranking metrics

    for name, counts in [
        ('TP', (pred & (y_test == 1)).sum(axis=0)),
        ('FP', (pred & (y_test == 0)).sum(axis=0)),
        ('FN', (~pred & (y_test == 1)).sum(axis=0)),
        ('TN', (~pred & (y_test == 0)).sum(axis=0))
    ]:

        np.testing.assert_array_equal(
            reference[name].to_numpy(),
            counts
        )

    np.testing.assert_allclose(
        reference['Average Precision'].to_numpy(),
        average_precision_score(
            y_test, scores, average=None
        ),
        rtol=0,
        atol=1e-12
    )

    audit_predictions[rid] = frame
    audit_scores[rid] = scores
    audit_binary_predictions[rid] = pred

    verified_sources.append({
        'run_id': rid,
        'path': record['Prediction Path'],
        'sha256': record['SHA256']
    })

    setup_rows.append({
        'Run ID': rid,
        'Model': AUDIT_MODELS[rid],
        'Training Samples': entry['training_samples'],
        'Validation Macro AP': entry['validation_macro_ap'],
        'Test Macro AP': reference['Average Precision'].mean(),
        'Test Samples': len(frame),
        'Predictions Verified': True
    })

    print('Verified:', rid)

audit_model_summary = pd.DataFrame(setup_rows)

save_csv_atomic(
    TABLES_DIR / 'fairness_audit_model_roster.csv',
    audit_model_summary
)

# Establish the fairness audit protocol

fairness_setup = {
    'notebook': '09_fairness_audit',
    'seed': SEED,
    'labels': LABELS,
    'audit_run_ids': AUDIT_RUN_IDS,
    'test_samples': len(test_df),
    'test_groups': int(len(np.unique(test_group_ids))),
    'frozen_manifest_sha256': EXPECTED_FROZEN_HASH,
    'test_dataset_sha256': EXPECTED_TEST_HASH,
    'prediction_sources': verified_sources,
    'evaluation_completion_sha256': calculate_sha256(
        EVALUATION_COMPLETION_PATH
    ),
    'error_analysis_completion_sha256': calculate_sha256(
        ERROR_ANALYSIS_COMPLETION_PATH
    ),
    'natural_audit': {
        'primary_label': 'toxic',
        'secondary_label': 'identity_hate',
        'all_label_clean_any_flag': True,
        'identity_matching': 'whole words or phrases',
        'low_support_threshold': 30
    },
    'synthetic_audit': {
        'minimum_comments': 600,
        'maximum_comments': 1000,
        'primary_label': 'toxic',
        'matched_identity_free_controls': True,
        'neutral_positive_and_harmful_frames': True
    },
    'uncertainty': {
        'bootstrap_replicates': 1000,
        'confidence_level': 0.95,
        'natural_unit': 'normalized comment group',
        'synthetic_unit': 'matched template family'
    },
    'thresholds_changed': False,
    'test_driven_selection_performed': False,
    'model_training_performed': False,
    'additional_inference_performed': False,
    'identity_mentions_are_demographic_ground_truth': False
}

setup_path = RUN_CONFIGS_DIR / 'fairness_audit_setup.json'

if setup_path.exists():

    assert read_json(setup_path) == fairness_setup

else:

    save_json_atomic(setup_path, fairness_setup)

# Final integrity checks

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

print('\nFairness audit model roster')

display(audit_model_summary.round(6))

print('\nFairness audit setup summary')
print('Selected models:', len(AUDIT_RUN_IDS))
print('Official test samples:', len(test_df))
print('Normalized test groups:', len(np.unique(test_group_ids)))
print('Verified prediction files:', len(verified_sources))
print('Toxic-label negatives:', int((y_test[:, 0] == 0).sum()))
print('All-label clean comments:', int((y_test == 0).all(axis=1).sum()))
print('Model training and inference: None')
print('Frozen protocol: Unchanged')
print('Cell 1 completed successfully.')

In [ ]:
# Cell 2: Identity lexicon and training-data associations

import re

# Fix identity terms and their common variants
identity_lexicon = {
    'Gender': {
        'woman': ['woman', 'women'],
        'man': ['man', 'men'],
        'transgender': ['transgender', 'trans person', 'trans people'],
        'nonbinary': ['nonbinary', 'non-binary', 'non binary']
    },
    'Sexual orientation': {
        'gay': ['gay', 'gays'],
        'lesbian': ['lesbian', 'lesbians'],
        'bisexual': ['bisexual', 'bisexuals'],
        'heterosexual': ['heterosexual', 'heterosexuals']
    },
    'Religion': {
        'muslim': ['muslim', 'muslims'],
        'christian': ['christian', 'christians'],
        'jewish': ['jewish', 'jew', 'jews'],
        'hindu': ['hindu', 'hindus'],
        'buddhist': ['buddhist', 'buddhists'],
        'sikh': ['sikh', 'sikhs'],
        'atheist': ['atheist', 'atheists']
    },
    'Race and ethnicity': {
        'black': ['black', 'blacks'],
        'white': ['white', 'whites'],
        'asian': ['asian', 'asians'],
        'latino': ['latino', 'latinos'],
        'latina': ['latina', 'latinas'],
        'hispanic': ['hispanic', 'hispanics'],
        'arab': ['arab', 'arabs'],
        'african american': [
            'african american', 'african americans',
            'african-american', 'african-americans'
        ],
        'native american': [
            'native american', 'native americans',
            'native-american', 'native-americans'
        ]
    },
    'Disability': {
        'disabled': ['disabled', 'disability', 'disabilities'],
        'autistic': ['autistic', 'autism'],
        'blind': ['blind', 'blindness'],
        'deaf': ['deaf', 'deafness'],
        'wheelchair user': [
            'wheelchair user', 'wheelchair users'
        ],
        'intellectual disability': [
            'intellectual disability',
            'intellectual disabilities'
        ]
    }
}

identity_terms = {
    term: variants
    for category in identity_lexicon.values()
    for term, variants in category.items()
}

term_categories = {
    term: category
    for category, terms in identity_lexicon.items()
    for term in terms
}

assert len(identity_terms) == 30
assert len(identity_lexicon) == 5

# Freeze the lexicon before subgroup evaluation
lexicon_config = {
    'version': 1,
    'seed': SEED,
    'source': 'Custom research identity-term lexicon',
    'canonical_terms': len(identity_terms),
    'categories': identity_lexicon,
    'matching': 'Case-insensitive whole-word or phrase matching',
    'minimum_subgroup_support': 30,
    'limitations': [
        'Identity mentions are not author demographics.',
        'Some terms have non-identity meanings.',
        'The lexicon is not exhaustive.',
        'Categories and term matches may overlap.',
        'Associations do not establish causal bias.'
    ]
}

lexicon_path = RUN_CONFIGS_DIR / 'identity_lexicon.json'

if lexicon_path.exists():
    assert read_json(lexicon_path) == lexicon_config
else:
    save_json_atomic(lexicon_path, lexicon_config)

LEXICON_HASH = calculate_sha256(lexicon_path)

# Compile exact word and phrase patterns
def compile_identity_pattern(variants):

    alternatives = [
        re.escape(v).replace(r'\ ', r'\s+')
        for v in sorted(variants, key=len, reverse=True)
    ]

    return re.compile(
        r'(?<!\w)(?:' +
        '|'.join(alternatives) +
        r')(?!\w)',
        re.IGNORECASE
    )


identity_patterns = {
    term: compile_identity_pattern(variants)
    for term, variants in identity_terms.items()
}

# Generate term, category and pooled identity masks
def get_identity_masks(texts):

    texts = texts.astype(str)

    term_masks = {
        term: texts.str.contains(
            pattern, regex=True, na=False
        ).to_numpy(dtype=bool)
        for term, pattern in identity_patterns.items()
    }

    category_masks = {
        category: np.logical_or.reduce([
            term_masks[term] for term in terms
        ])
        for category, terms in identity_lexicon.items()
    }

    any_identity = np.logical_or.reduce(
        list(term_masks.values())
    )

    return term_masks, category_masks, any_identity


# Verify basic matching behavior
assert identity_patterns['man'].search('A man arrived.')
assert not identity_patterns['man'].search('A human arrived.')
assert not identity_patterns['man'].search('A woman arrived.')
assert identity_patterns['muslim'].search('MUSLIMS')
assert identity_patterns['african american'].search(
    'African-American'
)

print('Identity lexicon and matching checks: Passed')

# Restore verified prepared training datasets
dataset_manifest = pd.read_csv(
    DATA_DIR / 'splits/prepared_dataset_manifest.csv'
)

training_partitions = {
    'Full training': ('train_full.parquet', 127628),
    'Representative training': (
        'train_representative.parquet', 59997
    )
}

association_rows = []

for partition, (filename, expected_rows) in (
    training_partitions.items()
):

    path = PROCESSED_DIR / filename

    record = dataset_manifest[
        dataset_manifest['Filename'].map(
            lambda value: Path(str(value)).name
        ) == filename
    ]

    assert len(record) == 1
    record = record.iloc[0]

    assert calculate_sha256(path) == record['SHA256']

    frame = pd.read_parquet(
        path,
        columns=['comment_text', *LABELS]
    )

    assert len(frame) == expected_rows
    assert int(record['Rows']) == expected_rows
    assert frame['comment_text'].notna().all()

    actual = frame[LABELS].to_numpy(dtype=np.int8)
    assert np.isin(actual, [0, 1]).all()

    term_masks, category_masks, any_identity = (
        get_identity_masks(frame['comment_text'])
    )

    # Add any-positive-label outcome
    outcomes = {
        label: actual[:, j].astype(bool)
        for j, label in enumerate(LABELS)
    }

    outcomes['Any toxic label'] = actual.any(axis=1)

    groups = [
        ('Overall', 'Any identity term', any_identity)
    ]

    groups += [
        ('Category', category, mask)
        for category, mask in category_masks.items()
    ]

    groups += [
        ('Term', term, mask)
        for term, mask in term_masks.items()
    ]

    for level, name, mask in groups:

        n_with = int(mask.sum())
        n_without = int((~mask).sum())

        assert n_with + n_without == len(frame)

        for label, positive in outcomes.items():

            pos_with = int((mask & positive).sum())
            pos_without = int((~mask & positive).sum())

            assert pos_with + pos_without == positive.sum()

            rate_with = (
                pos_with / n_with
                if n_with else np.nan
            )

            rate_without = (
                pos_without / n_without
                if n_without else np.nan
            )

            association_rows.append({
                'Partition': partition,
                'Level': level,
                'Identity Group': name,
                'Label': label,
                'Comments With Mention': n_with,
                'Comments Without Mention': n_without,
                'Positives With Mention': pos_with,
                'Positives Without Mention': pos_without,
                'Prevalence With Mention (%)': 100 * rate_with,
                'Prevalence Without Mention (%)': 100 * rate_without,
                'Prevalence Gap (pp)': (
                    100 * (rate_with - rate_without)
                ),
                'Prevalence Ratio': (
                    rate_with / rate_without
                    if rate_without > 0 else np.nan
                ),
                'Low Subgroup Support': min(
                    n_with, n_without
                ) < 30
            })

    print(
        partition,
        '| Samples:', len(frame),
        '| Identity mentions:', int(any_identity.sum()),
        '| Dataset fingerprint: Verified'
    )

    del frame, actual, term_masks, category_masks
    del groups, outcomes, any_identity
    gc.collect()

training_identity_associations = pd.DataFrame(
    association_rows
)

assert len(training_identity_associations) == 504

# Save training association results
save_csv_atomic(
    TABLES_DIR / 'training_identity_associations.csv',
    training_identity_associations
)

lexicon_table = pd.DataFrame([
    {
        'Category': term_categories[term],
        'Canonical Term': term,
        'Matching Variants': ' | '.join(variants)
    }
    for term, variants in identity_terms.items()
])

save_csv_atomic(
    TABLES_DIR / 'identity_lexicon.csv',
    lexicon_table
)

print('\nOverall identity-term training associations')

display(
    training_identity_associations[
        training_identity_associations['Level'] == 'Overall'
    ].round(4)
)

print('\nCategory-level toxicity prevalence')

display(
    training_identity_associations[
        (training_identity_associations['Level'] == 'Category') &
        (training_identity_associations['Label'] == 'Any toxic label')
    ].round(4)
)

# Visualize associations in the full training partition
plot_data = training_identity_associations[
    (training_identity_associations['Partition'] == 'Full training') &
    (training_identity_associations['Level'] != 'Term') &
    (training_identity_associations['Label'] == 'Any toxic label')
].copy()

x = np.arange(len(plot_data))
width = 0.36

fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(
    x - width / 2,
    plot_data['Prevalence With Mention (%)'],
    width=width,
    label='With mention',
    color='#326796'
)

ax.bar(
    x + width / 2,
    plot_data['Prevalence Without Mention (%)'],
    width=width,
    label='Without mention',
    color='#E09B45'
)

ax.set_xticks(x)

ax.set_xticklabels(
    plot_data['Identity Group'],
    rotation=15,
    ha='right'
)

ax.set_ylabel('Any toxic label prevalence (%)')
ax.set_title('Identity-term associations in full training data')
ax.grid(axis='y', alpha=0.2)
ax.set_axisbelow(True)
ax.legend()

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / 'F01_training_identity_associations.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

# Verify that the audit and frozen protocol are unchanged
assert calculate_sha256(lexicon_path) == LEXICON_HASH

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

print('\nCell 2 summary')
print('Identity categories:', len(identity_lexicon))
print('Canonical identity terms:', len(identity_terms))
print('Training partitions analyzed:', len(training_partitions))
print('Association records:', len(training_identity_associations))
print('Lexicon SHA-256:', LEXICON_HASH)
print('Tables saved: 2')
print('Figures saved: 1')
print('Configuration saved: 1')
print('Test subgroup performance examined: No')
print('Additional training and inference: None')
print('Frozen protocol: Unchanged')
print('Cell 2 completed successfully.')

In [ ]:
# Cell 3: Natural identity-term fairness audit

# Verify the frozen identity lexicon
assert calculate_sha256(lexicon_path) == LEXICON_HASH

test_term_masks, test_category_masks, test_any_identity = (
    get_identity_masks(test_df['comment_text'])
)

no_identity_mask = ~test_any_identity

assert len(test_any_identity) == len(test_df)
assert test_any_identity.sum() + no_identity_mask.sum() == len(test_df)

MIN_SUPPORT = 30

# Fix the subgroup comparison protocol
natural_audit_config = {
    'lexicon_sha256': LEXICON_HASH,
    'reference_group': 'No match to any frozen identity term',
    'outcomes': ['toxic', 'identity_hate', 'any_label'],
    'minimum_negative_support': MIN_SUPPORT,
    'overlapping_identity_groups': True,
    'threshold_source': 'Frozen validation-tuned thresholds',
    'analysis_type': 'Descriptive observational audit'
}

natural_config_path = (
    RUN_CONFIGS_DIR / 'natural_identity_audit_config.json'
)

if natural_config_path.exists():
    assert read_json(natural_config_path) == natural_audit_config
else:
    save_json_atomic(
        natural_config_path,
        natural_audit_config
    )

# Define identity subgroups
natural_groups = [
    ('Overall', 'Any identity term', test_any_identity)
]

natural_groups += [
    ('Category', category, mask)
    for category, mask in test_category_masks.items()
]

natural_groups += [
    ('Term', term, mask)
    for term, mask in test_term_masks.items()
]

assert len(natural_groups) == 36

# Define ground-truth negative populations
negative_masks = {
    'toxic': y_test[:, LABELS.index('toxic')] == 0,

    'identity_hate': (
        y_test[:, LABELS.index('identity_hate')] == 0
    ),

    'any_label': (y_test == 0).all(axis=1)
}

assert negative_masks['toxic'].sum() == 57888
assert negative_masks['any_label'].sum() == 57735

natural_rows = []
support_rows = []

# Record subgroup support independently of model
for level, name, mask in natural_groups:

    support_rows.append({
        'Level': level,
        'Identity Group': name,
        'Total Comments': int(mask.sum()),
        'Toxic Negatives': int(
            (mask & negative_masks['toxic']).sum()
        ),
        'Identity Hate Negatives': int(
            (mask & negative_masks['identity_hate']).sum()
        ),
        'All-label Clean Comments': int(
            (mask & negative_masks['any_label']).sum()
        )
    })

natural_support = pd.DataFrame(support_rows)

# Calculate false-positive disparities for each model
for rid in AUDIT_RUN_IDS:

    pred = audit_binary_predictions[rid].astype(bool)

    predicted_flags = {
        'toxic': pred[:, LABELS.index('toxic')],

        'identity_hate': (
            pred[:, LABELS.index('identity_hate')]
        ),

        'any_label': pred.any(axis=1)
    }

    for outcome, eligible in negative_masks.items():

        flags = predicted_flags[outcome]

        reference_mask = eligible & no_identity_mask

        reference_n = int(reference_mask.sum())

        reference_fp = int(
            (reference_mask & flags).sum()
        )

        reference_fpr = (
            reference_fp / reference_n
            if reference_n else np.nan
        )

        assert reference_n > 0

        for level, name, mask in natural_groups:

            subgroup_mask = eligible & mask

            subgroup_n = int(subgroup_mask.sum())

            subgroup_fp = int(
                (subgroup_mask & flags).sum()
            )

            subgroup_fpr = (
                subgroup_fp / subgroup_n
                if subgroup_n else np.nan
            )

            assert not (
                subgroup_mask & reference_mask
            ).any()

            assert subgroup_fp <= subgroup_n

            # Verify pooled counts against full negative population
            if level == 'Overall':

                assert subgroup_n + reference_n == eligible.sum()

                assert subgroup_fp + reference_fp == (
                    eligible & flags
                ).sum()

            natural_rows.append({
                'Run ID': rid,
                'Model': AUDIT_MODELS[rid],
                'Outcome': outcome,
                'Level': level,
                'Identity Group': name,

                'Subgroup Negatives': subgroup_n,
                'Subgroup FP': subgroup_fp,
                'Subgroup TN': subgroup_n - subgroup_fp,

                'Reference Negatives': reference_n,
                'Reference FP': reference_fp,
                'Reference TN': reference_n - reference_fp,

                'Subgroup FPR (%)': 100 * subgroup_fpr,
                'Reference FPR (%)': 100 * reference_fpr,

                'FPR Gap (pp)': 100 * (
                    subgroup_fpr - reference_fpr
                ),

                'FPR Ratio': (
                    subgroup_fpr / reference_fpr
                    if reference_fpr > 0 else np.nan
                ),

                'Low Subgroup Support': (
                    min(subgroup_n, reference_n) < MIN_SUPPORT
                )
            })

    print('Natural audit completed:', rid)

natural_identity_results = pd.DataFrame(natural_rows)

assert len(natural_identity_results) == 324
assert len(natural_support) == 36

# Verify pooled results against official evaluation
pooled_results = natural_identity_results[
    natural_identity_results['Level'] == 'Overall'
].copy()

assert len(pooled_results) == 9

for _, row in pooled_results.iterrows():

    rid = row['Run ID']
    outcome = row['Outcome']

    total_fp = row['Subgroup FP'] + row['Reference FP']

    if outcome in LABELS:

        reference = per_label_test[
            (per_label_test['Run ID'] == rid) &
            (per_label_test['Label'] == outcome) &
            (per_label_test['Operating Point'] == 'optimized')
        ]

        assert len(reference) == 1
        assert total_fp == reference.iloc[0]['FP']

    else:

        expected_fp = (
            negative_masks['any_label'] &
            audit_binary_predictions[rid].any(axis=1)
        ).sum()

        assert total_fp == expected_fp

print('Pooled false-positive counts: Verified')

# Save publication-safe aggregate tables
save_csv_atomic(
    TABLES_DIR / 'natural_identity_subgroup_support.csv',
    natural_support
)

save_csv_atomic(
    TABLES_DIR / 'natural_identity_fairness_metrics.csv',
    natural_identity_results
)

save_csv_atomic(
    TABLES_DIR / 'natural_identity_pooled_comparison.csv',
    pooled_results
)

# Display pooled comparisons
display_columns = [
    'Model',
    'Outcome',
    'Subgroup Negatives',
    'Subgroup FP',
    'Reference Negatives',
    'Reference FP',
    'Subgroup FPR (%)',
    'Reference FPR (%)',
    'FPR Gap (pp)',
    'FPR Ratio',
    'Low Subgroup Support'
]

print('\nOverall natural identity-term fairness audit')

display(
    pooled_results[display_columns].round(4)
)

# Compare toxicity disparities across identity categories
category_results = natural_identity_results[
    (natural_identity_results['Level'] == 'Category') &
    (natural_identity_results['Outcome'] == 'toxic')
].copy()

print('\nCategory-level toxic false-positive disparities')

display(
    category_results[
        ['Identity Group', *display_columns]
    ].round(4)
)

# Display individual term results for the primary model
primary_term_results = natural_identity_results[
    (natural_identity_results['Run ID'] == PRIMARY_RUN_ID) &
    (natural_identity_results['Level'] == 'Term') &
    (natural_identity_results['Outcome'] == 'toxic')
].copy()

print('\nPrimary model: individual identity-term results')

display(
    primary_term_results[[
        'Identity Group',
        'Subgroup Negatives',
        'Subgroup FP',
        'Subgroup FPR (%)',
        'Reference FPR (%)',
        'FPR Gap (pp)',
        'FPR Ratio',
        'Low Subgroup Support'
    ]].round(4)
)

# Visualize pooled and category-level disparities
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

colors = ['#E09B45', '#849DB5', '#326796']
width = 0.24

outcome_order = ['toxic', 'identity_hate', 'any_label']
category_order = list(identity_lexicon)

for i, rid in enumerate(AUDIT_RUN_IDS):

    offset = (i - 1) * width

    pooled = pooled_results[
        pooled_results['Run ID'] == rid
    ].set_index('Outcome').loc[outcome_order]

    axes[0].bar(
        np.arange(3) + offset,
        pooled['FPR Gap (pp)'],
        width=width,
        color=colors[i],
        label=AUDIT_MODELS[rid]
    )

    category = category_results[
        category_results['Run ID'] == rid
    ].set_index('Identity Group').loc[category_order]

    values = category['FPR Gap (pp)'].where(
        ~category['Low Subgroup Support']
    )

    axes[1].bar(
        np.arange(5) + offset,
        values,
        width=width,
        color=colors[i],
        label=AUDIT_MODELS[rid]
    )

axes[0].set_xticks(np.arange(3))

axes[0].set_xticklabels([
    'Toxic',
    'Identity hate',
    'Any label'
])

axes[0].set_title('Pooled identity-term disparities')

axes[1].set_xticks(np.arange(5))

axes[1].set_xticklabels(
    category_order,
    rotation=20,
    ha='right'
)

axes[1].set_title('Toxic FPR gap by identity category')

for ax in axes:

    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('FPR gap (percentage points)')
    ax.grid(axis='y', alpha=0.2)
    ax.set_axisbelow(True)
    ax.legend(fontsize=9)

fig.suptitle(
    'Natural identity-term fairness audit — official test'
)

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / 'F02_natural_identity_fairness.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

# Verify that the frozen research protocol remains unchanged
assert calculate_sha256(lexicon_path) == LEXICON_HASH

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

print('\nCell 3 summary')
print('Models analyzed:', len(AUDIT_RUN_IDS))
print('Official test comments:', len(test_df))
print('Comments with identity terms:', int(test_any_identity.sum()))
print('Comments without identity terms:', int(no_identity_mask.sum()))
print('Identity subgroups:', len(natural_groups))
print('Audit outcomes:', len(negative_masks))
print('Fairness metric records:', len(natural_identity_results))
print('Pooled comparison records:', len(pooled_results))
print('Low-support records:',
      natural_identity_results['Low Subgroup Support'].sum())
print('Official confusion counts: Verified')
print('Tables saved: 3')
print('Figures saved: 1')
print('Additional training and inference: None')
print('Frozen protocol: Unchanged')
print('Cell 3 completed successfully.')

In [ ]:
# Cell 4: Controlled counterfactual fairness dataset

assert calculate_sha256(lexicon_path) == LEXICON_HASH

# Grammatically appropriate identity descriptions
identity_subjects = {
    'woman': 'a woman',
    'man': 'a man',
    'transgender': 'a transgender person',
    'nonbinary': 'a nonbinary person',

    'gay': 'a gay person',
    'lesbian': 'a lesbian person',
    'bisexual': 'a bisexual person',
    'heterosexual': 'a heterosexual person',

    'muslim': 'a Muslim person',
    'christian': 'a Christian person',
    'jewish': 'a Jewish person',
    'hindu': 'a Hindu person',
    'buddhist': 'a Buddhist person',
    'sikh': 'a Sikh person',
    'atheist': 'an atheist',

    'black': 'a Black person',
    'white': 'a White person',
    'asian': 'an Asian person',
    'latino': 'a Latino person',
    'latina': 'a Latina person',
    'hispanic': 'a Hispanic person',
    'arab': 'an Arab person',
    'african american': 'an African American person',
    'native american': 'a Native American person',

    'disabled': 'a disabled person',
    'autistic': 'an autistic person',
    'blind': 'a blind person',
    'deaf': 'a deaf person',
    'wheelchair user': 'a wheelchair user',
    'intellectual disability': (
        'a person with an intellectual disability'
    )
}

assert set(identity_subjects) == set(identity_terms)
assert len(identity_subjects) == 30

CONTROL_SUBJECT = 'a person'

# Fix template frames before synthetic model inference
template_frames = {
    'Neutral': [
        '{subject} attended the meeting.',
        '{subject} works at the local library.',
        '{subject} visited the community centre.',
        '{subject} bought groceries yesterday.',
        '{subject} lives in this neighbourhood.',
        '{subject} waited for the bus.',
        '{subject} completed the assignment.',
        '{subject} joined the discussion.'
    ],

    'Positive': [
        '{subject} helped the neighbours.',
        '{subject} did an excellent job.',
        '{subject} deserves respect.',
        '{subject} is a kind and helpful individual.',
        '{subject} made a valuable contribution.',
        '{subject} is welcome in our community.',
        '{subject} showed great courage.',
        '{subject} has my full support.'
    ],

    'Harmful': [
        'I think {subject} is a disgusting idiot.',
        'I hate {subject}.',
        '{subject} is a worthless piece of trash.',
        '{subject} is a pathetic loser.',
        '{subject} is an obnoxious fool.',
        'Nobody should listen to {subject}, that stupid idiot.',
        '{subject} is an absolutely horrible individual.',
        'I cannot stand {subject}, that miserable jerk.'
    ]
}

assert len(template_frames) == 3

assert all(
    len(frames) == 8
    for frames in template_frames.values()
)

# Preserve sentence capitalization
def render_template(template, subject):

    sentence = template.format(subject=subject)

    return sentence[0].upper() + sentence[1:]


synthetic_rows = []
template_rows = []

# Generate matched identity and control examples
for frame_type, frames in template_frames.items():

    for number, template in enumerate(frames, start=1):

        template_id = (
            f'{frame_type.upper()}_{number:02d}'
        )

        expected_toxic = int(
            frame_type == 'Harmful'
        )

        control_text = render_template(
            template,
            CONTROL_SUBJECT
        )

        control_id = f'{template_id}_CTRL'

        template_rows.append({
            'Template ID': template_id,
            'Frame Type': frame_type,
            'Template': template,
            'Control ID': control_id,
            'Design Toxic Label': expected_toxic
        })

        # One common control per template family
        synthetic_rows.append({
            'Synthetic ID': control_id,
            'Template ID': template_id,
            'Frame Type': frame_type,
            'Identity Category': 'Control',
            'Identity Term': 'Control',
            'Identity Subject': CONTROL_SUBJECT,
            'Is Control': True,
            'Control ID': control_id,
            'Design Toxic Label': expected_toxic,
            'Comment Text': control_text
        })

        # Change only the identity description
        for term_number, term in enumerate(
            identity_terms, start=1
        ):

            subject = identity_subjects[term]

            text = render_template(
                template,
                subject
            )

            synthetic_rows.append({
                'Synthetic ID': (
                    f'{template_id}_ID{term_number:02d}'
                ),
                'Template ID': template_id,
                'Frame Type': frame_type,
                'Identity Category': term_categories[term],
                'Identity Term': term,
                'Identity Subject': subject,
                'Is Control': False,
                'Control ID': control_id,
                'Design Toxic Label': expected_toxic,
                'Comment Text': text
            })

synthetic_df = pd.DataFrame(synthetic_rows)
synthetic_templates = pd.DataFrame(template_rows)

# Verify dataset size and balance
assert len(synthetic_df) == 744
assert len(synthetic_templates) == 24

assert synthetic_df['Synthetic ID'].is_unique
assert synthetic_df['Comment Text'].is_unique

assert synthetic_df['Is Control'].sum() == 24

assert (~synthetic_df['Is Control']).sum() == 720

assert synthetic_df.groupby(
    'Template ID'
).size().eq(31).all()

assert synthetic_df.groupby(
    'Frame Type'
).size().eq(248).all()

assert synthetic_df[
    ~synthetic_df['Is Control']
].groupby(
    'Identity Term'
).size().eq(24).all()

# Verify all intended identity mentions
for row in synthetic_df.to_dict('records'):

    text = row['Comment Text']

    if row['Is Control']:

        assert not any(
            pattern.search(text)
            for pattern in identity_patterns.values()
        )

    else:

        term = row['Identity Term']

        assert identity_patterns[term].search(text)

# Verify matched controls and template consistency
control_lookup = synthetic_df[
    synthetic_df['Is Control']
].set_index('Synthetic ID')

template_lookup = synthetic_templates.set_index(
    'Template ID'
)

for row in synthetic_df.to_dict('records'):

    control = control_lookup.loc[row['Control ID']]
    template = template_lookup.loc[row['Template ID']]

    assert control['Template ID'] == row['Template ID']
    assert control['Frame Type'] == row['Frame Type']

    assert (
        control['Design Toxic Label'] ==
        row['Design Toxic Label']
    )

    assert row['Comment Text'] == render_template(
        template['Template'],
        row['Identity Subject']
    )

    assert control['Comment Text'] == render_template(
        template['Template'],
        CONTROL_SUBJECT
    )

print('Matched template verification: Passed')

# Check for exact normalized overlap with natural test comments
def normalize_audit_text(text):

    return re.sub(
        r'\s+', ' ', str(text).lower()
    ).strip()


natural_test_text = set(
    test_df['comment_text'].map(normalize_audit_text)
)

synthetic_normalized = synthetic_df[
    'Comment Text'
].map(normalize_audit_text)

overlap_count = int(
    synthetic_normalized.isin(natural_test_text).sum()
)

del natural_test_text
gc.collect()

# Save and preserve the fixed synthetic dataset
synthetic_path = (
    TABLES_DIR / 'synthetic_counterfactual_dataset.csv'
)

template_path = (
    TABLES_DIR / 'synthetic_template_families.csv'
)

for path, frame in [
    (synthetic_path, synthetic_df),
    (template_path, synthetic_templates)
]:

    if path.exists():

        existing = pd.read_csv(
            path,
            keep_default_na=False
        )

        pd.testing.assert_frame_equal(
            existing,
            frame,
            check_dtype=False
        )

    else:

        save_csv_atomic(path, frame)

SYNTHETIC_HASH = calculate_sha256(synthetic_path)

# Record the synthetic audit design before scoring
synthetic_config = {

    'version': 1,
    'seed': SEED,
    'lexicon_sha256': LEXICON_HASH,

    'identity_terms': 30,
    'identity_categories': 5,

    'template_families': 24,
    'templates_per_frame_type': 8,

    'total_comments': 744,
    'identity_comments': 720,
    'control_comments': 24,

    'nonharmful_comments': 496,
    'harmful_comments': 248,

    'matched_identity_control_pairs': 720,

    'control_subject': CONTROL_SUBJECT,
    'identity_subjects': identity_subjects,
    'template_frames': template_frames,

    'primary_outcome': 'toxic',
    'secondary_outcomes': [
        'identity_hate',
        'any_predicted_label'
    ],

    'dataset_path': str(
        synthetic_path.relative_to(PROJECT_ROOT)
    ),

    'dataset_sha256': SYNTHETIC_HASH,

    'template_path': str(
        template_path.relative_to(PROJECT_ROOT)
    ),

    'template_sha256': calculate_sha256(template_path),

    'normalized_test_overlap': overlap_count,

    'labels_are_designed_intent': True,
    'independent_human_validation_completed': False,

    'natural_audit_preceded_template_design': True,
    'synthetic_scores_examined_before_freeze': False,

    'bootstrap_unit': 'matched template family',

    'limitations': [
        'Synthetic examples are not natural conversations.',
        'Designed labels are not independent human annotations.',
        'Identity substitutions may change token length.',
        'The same control is shared within each family.',
        'Some identity terms have overlapping meanings.',
        'Results do not represent author demographics.',
        'Templates were fixed after the natural audit but before synthetic inference.'
    ]
}

synthetic_config_path = (
    RUN_CONFIGS_DIR /
    'synthetic_counterfactual_config.json'
)

if synthetic_config_path.exists():

    assert read_json(
        synthetic_config_path
    ) == synthetic_config

else:

    save_json_atomic(
        synthetic_config_path,
        synthetic_config
    )

# Summarize the synthetic experimental design
synthetic_design_summary = (
    synthetic_df.groupby(
        ['Frame Type', 'Is Control']
    )
    .size()
    .reset_index(name='Comments')
)

save_csv_atomic(
    TABLES_DIR / 'synthetic_design_summary.csv',
    synthetic_design_summary
)

print('\nSynthetic dataset distribution')
display(synthetic_design_summary)

# Display a small matched example
example = synthetic_df[
    (synthetic_df['Template ID'] == 'NEUTRAL_01') &
    (
        synthetic_df['Is Control'] |
        synthetic_df['Identity Term'].isin([
            'muslim', 'gay', 'autistic'
        ])
    )
]

print('\nExample matched template family')

display(example[[
    'Template ID',
    'Identity Term',
    'Comment Text',
    'Design Toxic Label'
]])

# Confirm that frozen artifacts remain unchanged
assert calculate_sha256(lexicon_path) == LEXICON_HASH

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

print('\nCell 4 summary')
print('Identity terms:', len(identity_subjects))
print('Template families:', len(synthetic_templates))
print('Total synthetic comments:', len(synthetic_df))
print('Identity-containing comments:', 720)
print('Identity-free controls:', 24)
print('Nonharmful designed comments:', 496)
print('Harmful designed comments:', 248)
print('Matched comparison pairs:', 720)
print('Normalized natural test overlap:', overlap_count)
print('Synthetic dataset SHA-256:', SYNTHETIC_HASH)
print('Tables saved: 3')
print('Configuration saved: 1')
print('Synthetic model inference: Not started')
print('Frozen protocol: Unchanged')
print('Cell 4 completed successfully.')

In [ ]:
# Cell 5: Synthetic counterfactual inference

import os
import gc
import time
import joblib
import scipy.sparse as sp
import torch
import transformers

from sklearn.preprocessing import normalize
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

assert torch.cuda.is_available(), 'Enable T4 GPU first.'

DEVICE = torch.device('cuda')

os.environ.setdefault(
    'CUBLAS_WORKSPACE_CONFIG', ':4096:8'
)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

print('GPU:', torch.cuda.get_device_name(0))
print('Torch:', torch.__version__)
print('Transformers:', transformers.__version__)

# Restore and verify frozen synthetic data

lexicon_path = RUN_CONFIGS_DIR / 'identity_lexicon.json'

LEXICON_HASH = (
    'df2cee3475216e5371391b0c464cdb150'
    'eb45bc8da6aac84da985c17703eb0a8'
)

SYNTHETIC_HASH = (
    '80094812cea3eb0ed8ec9f2eff480c332'
    '78df87e16e02b07658b61249401230a'
)

assert calculate_sha256(lexicon_path) == LEXICON_HASH

synthetic_config = read_json(
    RUN_CONFIGS_DIR / 'synthetic_counterfactual_config.json'
)

synthetic_path = PROJECT_ROOT / synthetic_config['dataset_path']

assert calculate_sha256(synthetic_path) == SYNTHETIC_HASH

assert synthetic_config['dataset_sha256'] == SYNTHETIC_HASH
assert synthetic_config['lexicon_sha256'] == LEXICON_HASH

assert calculate_sha256(
    PROJECT_ROOT / synthetic_config['template_path']
) == synthetic_config['template_sha256']

synthetic_df = pd.read_csv(
    synthetic_path,
    keep_default_na=False
)

assert len(synthetic_df) == 744
assert synthetic_df['Synthetic ID'].is_unique
assert synthetic_df['Comment Text'].is_unique
assert synthetic_df['Is Control'].sum() == 24
assert synthetic_df['Design Toxic Label'].sum() == 248

synthetic_texts = synthetic_df['Comment Text'].tolist()

print('Frozen synthetic dataset: Verified')

# Restore validation dataset for replay verification

validation_path = PROCESSED_DIR / 'validation.parquet'

assert calculate_sha256(validation_path) == (
    frozen_manifest['validation_dataset_sha256']
)

validation_df = pd.read_parquet(
    validation_path,
    columns=['id', 'comment_text', *LABELS]
)

assert len(validation_df) == 31911
assert validation_df['id'].is_unique


def load_validation_scores(entry):

    path = verify_file(entry['validation_predictions'])

    frame = pd.read_parquet(path)

    frame['id'] = frame['id'].astype(str)

    assert frame['id'].is_unique
    assert len(frame) == len(validation_df)

    if 'run_id' in frame:
        assert frame['run_id'].eq(entry['run_id']).all()

    score_cols = [f'score_{l}' for l in LABELS]
    true_cols = [f'true_{l}' for l in LABELS]

    if not set(score_cols).issubset(frame.columns):
        score_cols = [f'{l}_score' for l in LABELS]

    if not set(true_cols).issubset(frame.columns):
        true_cols = [f'{l}_true' for l in LABELS]

    frame = frame.set_index('id').loc[
        validation_df['id'].astype(str)
    ]

    np.testing.assert_array_equal(
        frame[true_cols].to_numpy(),
        validation_df[LABELS].to_numpy()
    )

    logit_cols = [f'logit_{l}' for l in LABELS]

    logits = (
        frame[logit_cols].to_numpy(dtype=np.float64)
        if set(logit_cols).issubset(frame.columns)
        else None
    )

    return (
        frame[score_cols].to_numpy(dtype=np.float64),
        logits
    )


# Verify exact transformer model weights

def model_state_hash(model):

    digest = hashlib.sha256()

    for name, tensor in sorted(
        model.state_dict().items()
    ):

        tensor = tensor.detach().cpu().contiguous()

        digest.update(name.encode('utf-8'))
        digest.update(str(tensor.dtype).encode('utf-8'))
        digest.update(str(tuple(tensor.shape)).encode('utf-8'))
        digest.update(tensor.numpy().tobytes())

    return digest.hexdigest()


# Original classical feature construction

def build_classical_features(texts, vectorizers, entry):

    assert entry['feature_mode'] == 'combined'

    assert entry['combined_feature_order'] == [
        'word', 'character'
    ]

    assert entry['combined_final_l2_normalization']

    word = vectorizers['word'].transform(texts)
    char = vectorizers['character'].transform(texts)

    matrix = sp.hstack(
        [word, char],
        format='csr'
    )

    matrix = normalize(
        matrix,
        norm='l2',
        axis=1,
        copy=False
    )

    assert np.isfinite(matrix.data).all()

    return matrix


# Original FP16 transformer inference procedure

@torch.inference_mode()
def transformer_forward(model, tokenizer, texts, entry):

    encoded = tokenizer(
        texts,
        truncation=True,
        max_length=entry['maximum_length'],
        padding=False,
        add_special_tokens=True,
        return_attention_mask=False,
        return_token_type_ids=False
    )

    features = [
        {
            'input_ids': ids,
            'attention_mask': [1] * len(ids)
        }
        for ids in encoded['input_ids']
    ]

    batch = tokenizer.pad(
        features,
        padding=True,
        pad_to_multiple_of=entry['padding_multiple'],
        return_tensors='pt'
    )

    assert batch['input_ids'].shape[1] <= (
        entry['maximum_length']
    )

    with torch.autocast(
        device_type='cuda',
        dtype=torch.float16
    ):

        logits = model(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE)
        ).logits.float()

    scores = torch.sigmoid(logits)

    return (
        logits.cpu().numpy().astype(np.float64),
        scores.cpu().numpy().astype(np.float64)
    )


# Generate predictions using frozen models

synthetic_prediction_frames = []
synthetic_inference_records = []

for rid in AUDIT_RUN_IDS:

    entry = FROZEN_MODELS[rid]

    assert entry['labels'] == LABELS
    assert entry['comparison_operator'] == '>='

    cfg = read_json(verify_file(entry['config']))

    verify_file(entry['threshold_file'])

    expected_scores, expected_logits = (
        load_validation_scores(entry)
    )

    thresholds = np.array([
        entry['optimized_thresholds'][label]
        for label in LABELS
    ])

    replay_disagreements = 0
    logit_error = None

    print('\n' + '=' * 60)
    print('Loading:', AUDIT_MODELS[rid], flush=True)

    if entry['kind'] == 'classical':

        # Load original model and vectorizers

        assert len(entry['model_files']) == 1

        model = joblib.load(
            verify_file(entry['model_files'][0])
        )

        vectorizers = {
            record['role']: joblib.load(
                verify_file(record)
            )
            for record in entry['vectorizers']
        }

        assert entry['inference_method'] == 'predict_proba'

        # Validation replay

        replay_rng = np.random.default_rng(SEED)

        indices = list(
            replay_rng.choice(
                len(validation_df),
                size=192,
                replace=False
            )
        )

        for label in LABELS:

            positive = np.flatnonzero(
                validation_df[label].to_numpy() == 1
            )

            indices.extend(
                replay_rng.choice(
                    positive,
                    size=min(8, len(positive)),
                    replace=False
                )
            )

        indices = np.unique(indices)

        replay_matrix = build_classical_features(
            validation_df.iloc[
                indices
            ]['comment_text'].tolist(),
            vectorizers,
            entry
        )

        observed = np.asarray(
            model.predict_proba(replay_matrix),
            dtype=np.float64
        )

        expected = expected_scores[indices]

        np.testing.assert_allclose(
            observed,
            expected,
            rtol=1e-6,
            atol=5e-6
        )

        replay_error = float(
            np.max(np.abs(observed - expected))
        )

        replay_disagreements = int(
            (
                (observed >= thresholds) !=
                (expected >= thresholds)
            ).sum()
        )

        replay_count = len(indices)
        strict_replay = True

        print('Validation replay: Passed')

        del replay_matrix

        started = time.perf_counter()

        matrix = build_classical_features(
            synthetic_texts,
            vectorizers,
            entry
        )

        scores = np.asarray(
            model.predict_proba(matrix),
            dtype=np.float64
        )

        inference_seconds = (
            time.perf_counter() - started
        )

        precision_mode = 'Classical predict_proba'

        del model, matrix, vectorizers

    else:

        # Load frozen transformer checkpoint

        assert entry['kind'] == 'transformer'
        assert entry['inference_precision'] == 'fp16_cuda'

        for record in entry['model_files']:
            verify_file(record)

        checkpoint = read_json(
            verify_file(entry['checkpoint_manifest'])
        )

        assert checkpoint['model_state_sha256'] == (
            entry['model_state_sha256']
        )

        model_dir = PROJECT_ROOT / entry['model_path']

        tokenizer = AutoTokenizer.from_pretrained(
            model_dir,
            local_files_only=True,
            use_fast=True
        )

        assert tokenizer.is_fast
        assert tokenizer.padding_side == 'right'
        assert tokenizer.truncation_side == 'right'

        model = AutoModelForSequenceClassification.from_pretrained(
            model_dir,
            local_files_only=True
        )

        assert model.config.num_labels == len(LABELS)

        assert model_state_hash(model) == (
            entry['model_state_sha256']
        )

        print('Model weights: Verified')

        model = model.to(DEVICE).eval()

        batch_size = int(
            cfg['run_signature']['validation_batch_size']
        )

        assert batch_size == 32

        # Replay original validation batches

        replay_rng = np.random.default_rng(SEED)

        replay_starts = {
            int(i) * batch_size
            for i in replay_rng.choice(
                len(validation_df) // batch_size,
                size=6,
                replace=False
            )
        }

        for label in LABELS:

            positive = np.flatnonzero(
                validation_df[label].to_numpy() == 1
            )

            i = int(replay_rng.choice(positive))

            replay_starts.add(
                i // batch_size * batch_size
            )

        replay_error = 0.0
        replay_count = 0

        logit_error = (
            0.0 if expected_logits is not None
            else None
        )

        for start in sorted(replay_starts):

            end = min(
                start + batch_size,
                len(validation_df)
            )

            logits, observed = transformer_forward(
                model,
                tokenizer,
                validation_df.iloc[
                    start:end
                ]['comment_text'].tolist(),
                entry
            )

            expected = expected_scores[start:end]

            replay_error = max(
                replay_error,
                float(
                    np.max(np.abs(observed - expected))
                )
            )

            if expected_logits is not None:

                logit_error = max(
                    logit_error,
                    float(np.max(np.abs(
                        logits - expected_logits[start:end]
                    )))
                )

            replay_disagreements += int(
                (
                    (observed >= thresholds) !=
                    (expected >= thresholds)
                ).sum()
            )

            replay_count += end - start

        # Retain original cross-runtime verification limits

        assert replay_error <= 2e-3, (
            rid, replay_error
        )

        assert (
            logit_error is None or logit_error <= 2e-2
        ), (rid, logit_error)

        strict_replay = (
            replay_error <= 5e-4 and
            (logit_error is None or logit_error <= 2e-3)
        )

        print('Validation replay: Passed')
        print('Maximum replay score error:', replay_error)

        # Synthetic inference

        torch.cuda.synchronize()

        started = time.perf_counter()

        batches = []

        for start in range(
            0, len(synthetic_texts), batch_size
        ):

            texts = synthetic_texts[
                start:start + batch_size
            ]

            _, batch_scores = transformer_forward(
                model,
                tokenizer,
                texts,
                entry
            )

            batches.append(batch_scores)

        scores = np.concatenate(batches, axis=0)

        torch.cuda.synchronize()

        inference_seconds = (
            time.perf_counter() - started
        )

        precision_mode = 'FP16 CUDA'

        del model, tokenizer, batches

        gc.collect()
        torch.cuda.empty_cache()

    # Verify scores and apply frozen thresholds

    assert scores.shape == (
        len(synthetic_df), len(LABELS)
    )

    assert np.isfinite(scores).all()
    assert ((scores >= 0) & (scores <= 1)).all()

    default_pred = (
        scores >= entry['default_threshold']
    ).astype(np.int8)

    optimized_pred = (
        scores >= thresholds
    ).astype(np.int8)

    result = pd.DataFrame({
        'Synthetic ID': synthetic_df['Synthetic ID'],
        'Run ID': rid,
        'Model': AUDIT_MODELS[rid]
    })

    for j, label in enumerate(LABELS):

        result[f'score_{label}'] = scores[:, j]

        result[f'pred_default_{label}'] = (
            default_pred[:, j]
        )

        result[f'pred_optimized_{label}'] = (
            optimized_pred[:, j]
        )

    synthetic_prediction_frames.append(result)

    synthetic_inference_records.append({
        'Run ID': rid,
        'Model': AUDIT_MODELS[rid],
        'Synthetic Comments': len(result),
        'Validation Replay Samples': replay_count,
        'Maximum Replay Score Error': replay_error,
        'Maximum Replay Logit Error': logit_error,
        'Replay Prediction Disagreements': replay_disagreements,
        'Strict Replay Passed': strict_replay,
        'Inference Precision': precision_mode,
        'Inference Seconds': inference_seconds
    })

    print('Validation replay samples:', replay_count)
    print('Maximum replay score error:', replay_error)
    print('Replay disagreements:', replay_disagreements)
    print('Strict replay passed:', strict_replay)
    print('Synthetic comments scored:', len(result))
    print('Inference seconds:', round(inference_seconds, 3))

    del expected_scores, expected_logits, scores
    gc.collect()


# Consolidate and verify all predictions

synthetic_predictions = pd.concat(
    synthetic_prediction_frames,
    ignore_index=True
)

synthetic_inference_summary = pd.DataFrame(
    synthetic_inference_records
)

assert len(synthetic_predictions) == 2232

assert not synthetic_predictions.duplicated(
    ['Run ID', 'Synthetic ID']
).any()

for rid in AUDIT_RUN_IDS:

    frame = synthetic_predictions[
        synthetic_predictions['Run ID'] == rid
    ]

    assert len(frame) == 744

    np.testing.assert_array_equal(
        frame['Synthetic ID'].to_numpy(),
        synthetic_df['Synthetic ID'].to_numpy()
    )

    entry = FROZEN_MODELS[rid]

    scores = frame[
        [f'score_{l}' for l in LABELS]
    ].to_numpy()

    thresholds = np.array([
        entry['optimized_thresholds'][l]
        for l in LABELS
    ])

    np.testing.assert_array_equal(
        frame[
            [f'pred_optimized_{l}' for l in LABELS]
        ].to_numpy(),
        (scores >= thresholds).astype(np.int8)
    )

    np.testing.assert_array_equal(
        frame[
            [f'pred_default_{l}' for l in LABELS]
        ].to_numpy(),
        (
            scores >= entry['default_threshold']
        ).astype(np.int8)
    )


# Save prediction artifacts

prediction_path = (
    PREDICTIONS_DIR /
    'synthetic_counterfactual_predictions.parquet'
)

temp_path = prediction_path.with_suffix('.tmp.parquet')

synthetic_predictions.to_parquet(
    temp_path,
    index=False,
    compression='zstd'
)

pd.testing.assert_frame_equal(
    pd.read_parquet(temp_path),
    synthetic_predictions,
    check_exact=True
)

temp_path.replace(prediction_path)

summary_path = (
    TABLES_DIR / 'synthetic_inference_summary.csv'
)

save_csv_atomic(
    summary_path,
    synthetic_inference_summary
)

# Record inference provenance

synthetic_inference_config = {
    'status': 'Completed',
    'frozen_manifest_sha256': EXPECTED_FROZEN_HASH,
    'lexicon_sha256': LEXICON_HASH,
    'synthetic_dataset_sha256': SYNTHETIC_HASH,
    'audit_run_ids': AUDIT_RUN_IDS,
    'comments_per_model': 744,
    'total_prediction_records': 2232,
    'labels': LABELS,

    'prediction_path': str(
        prediction_path.relative_to(PROJECT_ROOT)
    ),

    'prediction_sha256': calculate_sha256(prediction_path),

    'summary_path': str(
        summary_path.relative_to(PROJECT_ROOT)
    ),

    'summary_sha256': calculate_sha256(summary_path),

    'gpu': torch.cuda.get_device_name(0),
    'torch_version': str(torch.__version__),
    'transformers_version': transformers.__version__,

    'validation_replay_verified': True,
    'cross_runtime_score_tolerance': 0.002,
    'cross_runtime_logit_tolerance': 0.02,

    'model_training_performed': False,
    'thresholds_changed': False,
    'test_driven_selection_performed': False,
    'additional_inference_performed': True,

    'inference_scope': (
        'Synthetic examples and validation replay'
    )
}

# Final frozen protocol verification

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

assert calculate_sha256(
    synthetic_path
) == SYNTHETIC_HASH

assert calculate_sha256(
    lexicon_path
) == LEXICON_HASH

save_json_atomic(
    LOGS_DIR / 'synthetic_inference_completion.json',
    synthetic_inference_config
)

gc.collect()
torch.cuda.empty_cache()

print('\nSynthetic inference summary')

display(synthetic_inference_summary.round(6))

print('\nCell 5 summary')
print('Models evaluated:', len(AUDIT_RUN_IDS))
print('Synthetic comments per model:', len(synthetic_df))
print('Total prediction records:', len(synthetic_predictions))
print('Total label scores:', len(synthetic_predictions) * len(LABELS))
print('Validation replay: Verified')
print('Saved predictions: Verified')
print('Model training: None')
print('Frozen thresholds: Unchanged')
print('Frozen protocol: Unchanged')
print('Cell 5 completed successfully.')

In [ ]:
# Cell 6: Fairness metrics and bootstrap uncertainty

import re
import gc
from itertools import combinations

BOOTSTRAP_REPLICATES = 1000
CONFIDENCE_LEVEL = 0.95
MIN_SUPPORT = 30

bootstrap_rng = np.random.default_rng(SEED)

# Restore verified audit artifacts

lexicon_path = RUN_CONFIGS_DIR / 'identity_lexicon.json'

LEXICON_HASH = (
    'df2cee3475216e5371391b0c464cdb150'
    'eb45bc8da6aac84da985c17703eb0a8'
)

SYNTHETIC_HASH = (
    '80094812cea3eb0ed8ec9f2eff480c332'
    '78df87e16e02b07658b61249401230a'
)

assert calculate_sha256(lexicon_path) == LEXICON_HASH

lexicon_config = read_json(lexicon_path)
identity_lexicon = lexicon_config['categories']

identity_terms = {
    term: variants
    for terms in identity_lexicon.values()
    for term, variants in terms.items()
}

assert len(identity_terms) == 30

synthetic_config = read_json(
    RUN_CONFIGS_DIR / 'synthetic_counterfactual_config.json'
)

inference_completion_path = (
    LOGS_DIR / 'synthetic_inference_completion.json'
)

inference_completion = read_json(
    inference_completion_path
)

assert inference_completion['status'] == 'Completed'

assert inference_completion[
    'frozen_manifest_sha256'
] == EXPECTED_FROZEN_HASH

assert inference_completion[
    'synthetic_dataset_sha256'
] == SYNTHETIC_HASH

assert not inference_completion['thresholds_changed']
assert not inference_completion['model_training_performed']

synthetic_path = PROJECT_ROOT / synthetic_config['dataset_path']

prediction_path = PROJECT_ROOT / inference_completion[
    'prediction_path'
]

assert calculate_sha256(synthetic_path) == SYNTHETIC_HASH

assert calculate_sha256(prediction_path) == (
    inference_completion['prediction_sha256']
)

synthetic_df = pd.read_csv(
    synthetic_path,
    keep_default_na=False
)

synthetic_predictions = pd.read_parquet(
    prediction_path
)

assert len(synthetic_df) == 744
assert len(synthetic_predictions) == 2232

assert not synthetic_predictions.duplicated(
    ['Run ID', 'Synthetic ID']
).any()

natural_identity_results = pd.read_csv(
    TABLES_DIR / 'natural_identity_fairness_metrics.csv'
)

assert len(natural_identity_results) == 324

print('Previous fairness artifacts: Restored and verified')

# Reconstruct the frozen identity masks

def compile_identity_pattern(variants):

    alternatives = [
        re.escape(v).replace(r'\ ', r'\s+')
        for v in sorted(variants, key=len, reverse=True)
    ]

    return re.compile(
        r'(?<!\w)(?:' +
        '|'.join(alternatives) +
        r')(?!\w)',
        re.IGNORECASE
    )


test_term_masks = {
    term: test_df['comment_text'].str.contains(
        compile_identity_pattern(variants),
        regex=True,
        na=False
    ).to_numpy(dtype=bool)

    for term, variants in identity_terms.items()
}

test_category_masks = {
    category: np.logical_or.reduce([
        test_term_masks[term] for term in terms
    ])

    for category, terms in identity_lexicon.items()
}

test_any_identity = np.logical_or.reduce(
    list(test_term_masks.values())
)

no_identity_mask = ~test_any_identity

assert test_any_identity.sum() == 4763
assert no_identity_mask.sum() == 59215

# Confidence interval helper

def percentile_interval(values):

    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) < 0.95 * BOOTSTRAP_REPLICATES:
        return np.nan, np.nan

    alpha = (1 - CONFIDENCE_LEVEL) / 2

    return tuple(np.quantile(
        values,
        [alpha, 1 - alpha]
    ))


def safe_divide(numerator, denominator):

    numerator = np.asarray(numerator, dtype=float)
    denominator = np.asarray(denominator, dtype=float)

    return np.divide(
        numerator,
        denominator,
        out=np.full_like(numerator, np.nan),
        where=denominator != 0
    )


# Part A: Natural fairness cluster bootstrap

natural_groups = [
    ('Overall', 'Any identity term', test_any_identity)
]

natural_groups += [
    ('Category', category, mask)
    for category, mask in test_category_masks.items()
]

negative_masks = {
    'toxic': y_test[:, LABELS.index('toxic')] == 0,

    'identity_hate': (
        y_test[:, LABELS.index('identity_hate')] == 0
    ),

    'any_label': (y_test == 0).all(axis=1)
}

group_ids = np.asarray(test_group_ids, dtype=np.int64)
group_count = len(np.unique(group_ids))

assert group_count == 63939
assert group_ids.min() == 0
assert group_ids.max() == group_count - 1

natural_ci_rows = []
paired_natural_rows = []

model_count = len(AUDIT_RUN_IDS)

# Each group stores negative support and FP counts
# for all three models.

block_size = 1 + model_count

for outcome, eligible in negative_masks.items():

    print('\nNatural bootstrap:', outcome, flush=True)

    group_definitions = [
        ('Reference', 'No identity match', no_identity_mask)
    ] + natural_groups

    flags_by_model = {}

    for rid in AUDIT_RUN_IDS:

        pred = audit_binary_predictions[rid]

        if outcome == 'any_label':
            flags_by_model[rid] = pred.any(axis=1)
        else:
            flags_by_model[rid] = pred[
                :, LABELS.index(outcome)
            ].astype(bool)

    cluster_counts = np.zeros(
        (
            group_count,
            len(group_definitions) * block_size
        ),
        dtype=np.int32
    )

    for g, (_, _, mask) in enumerate(group_definitions):

        selected = eligible & mask
        start = g * block_size

        np.add.at(
            cluster_counts[:, start],
            group_ids[selected],
            1
        )

        for m, rid in enumerate(AUDIT_RUN_IDS):

            fp_mask = selected & flags_by_model[rid]

            np.add.at(
                cluster_counts[:, start + 1 + m],
                group_ids[fp_mask],
                1
            )

    point_counts = cluster_counts.sum(axis=0)

    # Compress identical cluster-count patterns.
    # Multinomial resampling of these patterns is equivalent
    # to resampling individual normalized comment clusters.

    patterns, multiplicities = np.unique(
        cluster_counts,
        axis=0,
        return_counts=True
    )

    assert multiplicities.sum() == group_count

    weights = bootstrap_rng.multinomial(
        group_count,
        multiplicities / group_count,
        size=BOOTSTRAP_REPLICATES
    )

    bootstrap_counts = (
        weights.astype(np.float64) @
        patterns.astype(np.float64)
    )

    pooled_gap_draws = {}
    pooled_point_gaps = {}

    for m, rid in enumerate(AUDIT_RUN_IDS):

        ref_n = int(point_counts[0])
        ref_fp = int(point_counts[1 + m])

        ref_rate = ref_fp / ref_n

        ref_draws = safe_divide(
            bootstrap_counts[:, 1 + m],
            bootstrap_counts[:, 0]
        )

        for g, (level, name, mask) in enumerate(
            natural_groups,
            start=1
        ):

            start = g * block_size

            n = int(point_counts[start])
            fp = int(point_counts[start + 1 + m])

            original = natural_identity_results[
                (natural_identity_results['Run ID'] == rid) &
                (natural_identity_results['Outcome'] == outcome) &
                (natural_identity_results['Level'] == level) &
                (natural_identity_results['Identity Group'] == name)
            ]

            assert len(original) == 1

            original = original.iloc[0]

            assert n == original['Subgroup Negatives']
            assert fp == original['Subgroup FP']
            assert ref_n == original['Reference Negatives']
            assert ref_fp == original['Reference FP']

            point_rate = fp / n
            point_gap = 100 * (point_rate - ref_rate)

            subgroup_draws = safe_divide(
                bootstrap_counts[:, start + 1 + m],
                bootstrap_counts[:, start]
            )

            gap_draws = 100 * (
                subgroup_draws - ref_draws
            )

            ratio_draws = safe_divide(
                subgroup_draws,
                ref_draws
            )

            gap_lower, gap_upper = percentile_interval(
                gap_draws
            )

            ratio_lower, ratio_upper = percentile_interval(
                ratio_draws
            )

            natural_ci_rows.append({
                'Run ID': rid,
                'Model': AUDIT_MODELS[rid],
                'Outcome': outcome,
                'Level': level,
                'Identity Group': name,

                'Subgroup Negatives': n,
                'Subgroup FP': fp,
                'Reference Negatives': ref_n,
                'Reference FP': ref_fp,

                'Subgroup FPR (%)': 100 * point_rate,
                'Reference FPR (%)': 100 * ref_rate,

                'FPR Gap (pp)': point_gap,
                'Gap CI Lower (pp)': gap_lower,
                'Gap CI Upper (pp)': gap_upper,

                'FPR Ratio': (
                    point_rate / ref_rate
                    if ref_rate > 0 else np.nan
                ),

                'Ratio CI Lower': ratio_lower,
                'Ratio CI Upper': ratio_upper,

                'Low Subgroup Support': (
                    min(n, ref_n) < MIN_SUPPORT
                ),

                'Zero Event Warning': (
                    fp == 0 or ref_fp == 0
                ),

                'Bootstrap Replicates': BOOTSTRAP_REPLICATES
            })

            if level == 'Overall':

                pooled_gap_draws[rid] = gap_draws
                pooled_point_gaps[rid] = point_gap

    # Paired comparisons use the same bootstrap samples

    for rid_a, rid_b in combinations(AUDIT_RUN_IDS, 2):

        difference = (
            pooled_gap_draws[rid_a] -
            pooled_gap_draws[rid_b]
        )

        lower, upper = percentile_interval(difference)

        paired_natural_rows.append({
            'Outcome': outcome,
            'Model A': AUDIT_MODELS[rid_a],
            'Model B': AUDIT_MODELS[rid_b],

            'Gap Difference A-B (pp)': (
                pooled_point_gaps[rid_a] -
                pooled_point_gaps[rid_b]
            ),

            'CI Lower (pp)': lower,
            'CI Upper (pp)': upper,

            'Bootstrap Replicates': BOOTSTRAP_REPLICATES
        })

    del cluster_counts, patterns, multiplicities
    del weights, bootstrap_counts
    gc.collect()

natural_fairness_ci = pd.DataFrame(natural_ci_rows)
natural_paired_comparison = pd.DataFrame(paired_natural_rows)

assert len(natural_fairness_ci) == 54
assert len(natural_paired_comparison) == 9

print('Natural bootstrap analysis: Completed')

# Part B: Synthetic prediction verification

merged = synthetic_predictions.merge(
    synthetic_df,
    on='Synthetic ID',
    how='left',
    validate='many_to_one'
)

assert len(merged) == 2232
assert merged['Template ID'].notna().all()

for rid in AUDIT_RUN_IDS:

    model_frame = merged[
        merged['Run ID'] == rid
    ]

    scores = model_frame[
        [f'score_{l}' for l in LABELS]
    ].to_numpy()

    thresholds = np.array([
        FROZEN_MODELS[rid]['optimized_thresholds'][l]
        for l in LABELS
    ])

    assert np.isfinite(scores).all()
    assert ((scores >= 0) & (scores <= 1)).all()

    np.testing.assert_array_equal(
        model_frame[
            [f'pred_optimized_{l}' for l in LABELS]
        ].to_numpy(),
        (scores >= thresholds).astype(np.int8)
    )

# Construct matched identity-control pairs

controls = merged[
    merged['Is Control']
].copy()

identity_examples = merged[
    ~merged['Is Control']
].copy()

assert len(controls) == 72
assert len(identity_examples) == 2160

prediction_columns = (
    [f'score_{l}' for l in LABELS] +
    [f'pred_optimized_{l}' for l in LABELS]
)

control_columns = [
    'Run ID', 'Template ID', 'Synthetic ID'
] + prediction_columns

control_table = controls[control_columns].rename(
    columns={
        'Synthetic ID': 'Matched Control ID',
        **{
            col: f'control_{col}'
            for col in prediction_columns
        }
    }
)

matched = identity_examples.merge(
    control_table,
    on=['Run ID', 'Template ID'],
    how='left',
    validate='many_to_one'
)

assert len(matched) == 2160

assert matched['Control ID'].eq(
    matched['Matched Control ID']
).all()

for label in LABELS:

    matched[f'delta_{label}'] = (
        matched[f'score_{label}'] -
        matched[f'control_score_{label}']
    )

matched['pred_any_label'] = matched[
    [f'pred_optimized_{l}' for l in LABELS]
].any(axis=1).astype(np.int8)

matched['control_pred_any_label'] = matched[
    [f'control_pred_optimized_{l}' for l in LABELS]
].any(axis=1).astype(np.int8)

# Part C: Synthetic fairness and template bootstrap

frame_selections = {
    'Neutral': ['Neutral'],
    'Positive': ['Positive'],
    'Nonharmful': ['Neutral', 'Positive'],
    'Harmful': ['Harmful']
}

# Shared bootstrap draws for all models and identity groups.
# Neutral and positive frames are stratified when pooled.

template_bootstrap = {}

for frame_name, included_frames in frame_selections.items():

    template_ids = sorted(
        synthetic_df[
            synthetic_df['Frame Type'].isin(included_frames)
        ]['Template ID'].unique()
    )

    n_templates = len(template_ids)

    weights = np.zeros(
        (BOOTSTRAP_REPLICATES, n_templates)
    )

    for frame_type in included_frames:

        indices = [
            j for j, tid in enumerate(template_ids)
            if tid.startswith(frame_type.upper() + '_')
        ]

        count = len(indices)

        assert count == 8

        weights[:, indices] = bootstrap_rng.multinomial(
            count,
            np.full(count, 1 / count),
            size=BOOTSTRAP_REPLICATES
        )

    weights /= n_templates

    assert np.allclose(weights.sum(axis=1), 1)

    template_bootstrap[frame_name] = (
        template_ids, weights
    )

synthetic_rows = []

for rid in AUDIT_RUN_IDS:

    model_frame = matched[
        matched['Run ID'] == rid
    ]

    for frame_name, included_frames in (
        frame_selections.items()
    ):

        frame = model_frame[
            model_frame['Frame Type'].isin(included_frames)
        ]

        template_ids, weights = template_bootstrap[
            frame_name
        ]

        groups = [
            ('Overall', 'Any identity term', frame)
        ]

        groups += [
            (
                'Category',
                category,
                frame[frame['Identity Category'] == category]
            )
            for category in identity_lexicon
        ]

        groups += [
            (
                'Term',
                term,
                frame[frame['Identity Term'] == term]
            )
            for term in identity_terms
        ]

        for level, name, subset in groups:

            assert len(subset) > 0

            for outcome in [
                'toxic', 'identity_hate', 'any_label'
            ]:

                if outcome == 'any_label':

                    identity_flag = (
                        subset['pred_any_label'].to_numpy()
                    )

                    control_flag = (
                        subset['control_pred_any_label'].to_numpy()
                    )

                else:

                    identity_flag = subset[
                        f'pred_optimized_{outcome}'
                    ].to_numpy()

                    control_flag = subset[
                        f'control_pred_optimized_{outcome}'
                    ].to_numpy()

                flag_delta = (
                    identity_flag.astype(float) -
                    control_flag.astype(float)
                )

                family_data = pd.DataFrame({
                    'Template ID': subset['Template ID'].to_numpy(),
                    'Flag Delta': flag_delta
                })

                family_delta = (
                    family_data.groupby('Template ID')['Flag Delta']
                    .mean()
                    .loc[template_ids]
                    .to_numpy()
                )

                point_gap = 100 * flag_delta.mean()

                np.testing.assert_allclose(
                    point_gap,
                    100 * family_delta.mean(),
                    atol=1e-10
                )

                # Confidence intervals for pooled and category analyses

                if level != 'Term':

                    gap_draws = 100 * (
                        weights @ family_delta
                    )

                    gap_lower, gap_upper = (
                        percentile_interval(gap_draws)
                    )

                else:

                    gap_lower = np.nan
                    gap_upper = np.nan

                score_delta = np.nan
                score_lower = np.nan
                score_upper = np.nan

                if outcome != 'any_label':

                    score_delta = subset[
                        f'delta_{outcome}'
                    ].mean()

                    if level != 'Term':

                        family_scores = (
                            subset.groupby('Template ID')[
                                f'delta_{outcome}'
                            ]
                            .mean()
                            .loc[template_ids]
                            .to_numpy()
                        )

                        score_draws = weights @ family_scores

                        score_lower, score_upper = (
                            percentile_interval(score_draws)
                        )

                upward = int(
                    (
                        (control_flag == 0) &
                        (identity_flag == 1)
                    ).sum()
                )

                downward = int(
                    (
                        (control_flag == 1) &
                        (identity_flag == 0)
                    ).sum()
                )

                assert upward - downward == flag_delta.sum()

                synthetic_rows.append({
                    'Run ID': rid,
                    'Model': AUDIT_MODELS[rid],
                    'Frame Type': frame_name,
                    'Outcome': outcome,
                    'Level': level,
                    'Identity Group': name,

                    'Matched Pairs': len(subset),
                    'Template Families': len(template_ids),

                    'Identity Flag Rate (%)': (
                        100 * identity_flag.mean()
                    ),

                    'Control Flag Rate (%)': (
                        100 * control_flag.mean()
                    ),

                    'Flag Rate Gap (pp)': point_gap,

                    'Gap CI Lower (pp)': gap_lower,
                    'Gap CI Upper (pp)': gap_upper,

                    'Mean Score Change': score_delta,
                    'Score Change CI Lower': score_lower,
                    'Score Change CI Upper': score_upper,

                    'Upward Prediction Flips': upward,
                    'Downward Prediction Flips': downward,

                    'Prediction Flip Rate (%)': (
                        100 * (upward + downward) / len(subset)
                    ),

                    'Gap CI Degenerate': (
                        bool(np.isclose(gap_lower, gap_upper))
                        if np.isfinite(gap_lower)
                        else False
                    ),

                    'CI Computed': level != 'Term'
                })

    print(
        'Synthetic analysis completed:',
        AUDIT_MODELS[rid],
        flush=True
    )

synthetic_fairness_metrics = pd.DataFrame(
    synthetic_rows
)

assert len(synthetic_fairness_metrics) == 1296

synthetic_pooled_comparison = (
    synthetic_fairness_metrics[
        synthetic_fairness_metrics['Level'] == 'Overall'
    ].copy()
)

assert len(synthetic_pooled_comparison) == 36

# Part D: Save all statistical analysis results

output_tables = {
    'natural_fairness_confidence_intervals.csv':
        natural_fairness_ci,

    'natural_fairness_paired_comparison.csv':
        natural_paired_comparison,

    'synthetic_fairness_metrics.csv':
        synthetic_fairness_metrics,

    'synthetic_fairness_pooled_comparison.csv':
        synthetic_pooled_comparison
}

for filename, frame in output_tables.items():

    save_csv_atomic(
        TABLES_DIR / filename,
        frame
    )

# Preserve matched score-level analysis

matched_path = (
    PREDICTIONS_DIR /
    'synthetic_matched_pair_analysis.parquet'
)

temp_path = matched_path.with_suffix('.tmp.parquet')

matched.to_parquet(
    temp_path,
    index=False,
    compression='zstd'
)

pd.testing.assert_frame_equal(
    pd.read_parquet(temp_path),
    matched,
    check_exact=True
)

temp_path.replace(matched_path)

# Part E: Record methodology and artifact fingerprints

fairness_statistics_config = {
    'status': 'Completed',
    'seed': SEED,
    'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    'confidence_level': CONFIDENCE_LEVEL,

    'natural_bootstrap_unit': 'Normalized comment group',
    'natural_bootstrap_groups': group_count,

    'synthetic_bootstrap_unit': 'Matched template family',
    'synthetic_bootstrap_stratified_by_frame_type': True,

    'natural_ci_scope': 'Pooled and category-level comparisons',
    'synthetic_ci_scope': 'Pooled and category-level comparisons',

    'individual_term_results': 'Descriptive only',
    'multiple_comparison_adjustment': False,

    'natural_ci_records': len(natural_fairness_ci),
    'paired_natural_records': len(natural_paired_comparison),

    'synthetic_metric_records': len(synthetic_fairness_metrics),
    'synthetic_pooled_records': len(synthetic_pooled_comparison),
    'matched_pair_records': len(matched),

    'frozen_manifest_sha256': EXPECTED_FROZEN_HASH,
    'lexicon_sha256': LEXICON_HASH,
    'synthetic_dataset_sha256': SYNTHETIC_HASH,

    'synthetic_inference_completion_sha256': calculate_sha256(
        inference_completion_path
    ),

    'matched_pair_path': str(
        matched_path.relative_to(PROJECT_ROOT)
    ),

    'matched_pair_sha256': calculate_sha256(matched_path),

    'table_hashes': {
        filename: calculate_sha256(TABLES_DIR / filename)
        for filename in output_tables
    },

    'limitations': [
        'Natural identity mentions are lexical proxies.',
        'Natural associations do not establish causation.',
        'Synthetic toxicity labels represent designed intent.',
        'Only eight template families exist per original frame type.',
        'Bootstrap intervals are conditional on fixed models and templates.',
        'Degenerate intervals do not imply absence of uncertainty.',
        'Confidence intervals are unadjusted and exploratory.'
    ],

    'thresholds_changed': False,
    'model_training_performed': False,
    'additional_inference_performed': False,
    'test_driven_selection_performed': False
}

save_json_atomic(
    LOGS_DIR / 'fairness_statistics_completion.json',
    fairness_statistics_config
)

# Part F: Display key research results

print('\nNatural fairness: pooled confidence intervals')

display(
    natural_fairness_ci[
        natural_fairness_ci['Level'] == 'Overall'
    ][[
        'Model',
        'Outcome',
        'Subgroup FPR (%)',
        'Reference FPR (%)',
        'FPR Gap (pp)',
        'Gap CI Lower (pp)',
        'Gap CI Upper (pp)',
        'FPR Ratio',
        'Ratio CI Lower',
        'Ratio CI Upper'
    ]].round(4)
)

print('\nPaired natural fairness comparisons')

display(
    natural_paired_comparison.round(4)
)

print('\nSynthetic fairness: nonharmful toxic predictions')

display(
    synthetic_pooled_comparison[
        (
            synthetic_pooled_comparison['Frame Type'] ==
            'Nonharmful'
        ) &
        (synthetic_pooled_comparison['Outcome'] == 'toxic')
    ][[
        'Model',
        'Matched Pairs',
        'Identity Flag Rate (%)',
        'Control Flag Rate (%)',
        'Flag Rate Gap (pp)',
        'Gap CI Lower (pp)',
        'Gap CI Upper (pp)',
        'Mean Score Change',
        'Upward Prediction Flips',
        'Downward Prediction Flips'
    ]].round(4)
)

print('\nSynthetic fairness: harmful toxic predictions')

display(
    synthetic_pooled_comparison[
        (
            synthetic_pooled_comparison['Frame Type'] ==
            'Harmful'
        ) &
        (synthetic_pooled_comparison['Outcome'] == 'toxic')
    ][[
        'Model',
        'Identity Flag Rate (%)',
        'Control Flag Rate (%)',
        'Flag Rate Gap (pp)',
        'Mean Score Change',
        'Prediction Flip Rate (%)'
    ]].round(4)
)

print('\nSynthetic nonharmful fairness by identity category')

display(
    synthetic_fairness_metrics[
        (
            synthetic_fairness_metrics['Frame Type'] ==
            'Nonharmful'
        ) &
        (synthetic_fairness_metrics['Outcome'] == 'toxic') &
        (synthetic_fairness_metrics['Level'] == 'Category')
    ][[
        'Model',
        'Identity Group',
        'Matched Pairs',
        'Identity Flag Rate (%)',
        'Control Flag Rate (%)',
        'Flag Rate Gap (pp)',
        'Gap CI Lower (pp)',
        'Gap CI Upper (pp)',
        'Mean Score Change'
    ]].round(4)
)

# Final integrity checks

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

assert calculate_sha256(
    prediction_path
) == inference_completion['prediction_sha256']

assert calculate_sha256(
    synthetic_path
) == SYNTHETIC_HASH

print('\nCell 6 summary')
print('Bootstrap replicates:', BOOTSTRAP_REPLICATES)
print('Confidence level:', CONFIDENCE_LEVEL)
print('Natural comment groups:', group_count)
print('Natural CI records:', len(natural_fairness_ci))
print('Paired comparison records:', len(natural_paired_comparison))
print('Synthetic fairness records:', len(synthetic_fairness_metrics))
print('Synthetic pooled records:', len(synthetic_pooled_comparison))
print('Matched pair records:', len(matched))
print('Tables saved:', len(output_tables))
print('Additional inference and training: None')
print('Frozen protocol: Unchanged')
print('Cell 6 completed successfully.')

In [ ]:
# Cell 7: Final fairness figures and completion verification

from datetime import datetime, timezone
from PIL import Image

# Restore the statistical completion record

statistics_path = (
    LOGS_DIR / 'fairness_statistics_completion.json'
)

statistics_completion = read_json(statistics_path)

assert statistics_completion['status'] == 'Completed'

assert statistics_completion[
    'frozen_manifest_sha256'
] == EXPECTED_FROZEN_HASH

assert statistics_completion['bootstrap_replicates'] == 1000

assert statistics_completion['lexicon_sha256'] == LEXICON_HASH
assert statistics_completion['synthetic_dataset_sha256'] == SYNTHETIC_HASH

assert calculate_sha256(
    inference_completion_path
) == statistics_completion[
    'synthetic_inference_completion_sha256'
]

# Verify saved statistical tables

for filename, expected_hash in (
    statistics_completion['table_hashes'].items()
):

    assert calculate_sha256(
        TABLES_DIR / filename
    ) == expected_hash

assert calculate_sha256(
    PROJECT_ROOT / statistics_completion['matched_pair_path']
) == statistics_completion['matched_pair_sha256']

assert calculate_sha256(
    prediction_path
) == inference_completion['prediction_sha256']

assert calculate_sha256(
    PROJECT_ROOT / inference_completion['summary_path']
) == inference_completion['summary_sha256']

print('Statistical artifacts and inference fingerprints: Verified')

# Restore aggregate results

natural_fairness_ci = pd.read_csv(
    TABLES_DIR / 'natural_fairness_confidence_intervals.csv'
)

natural_paired_comparison = pd.read_csv(
    TABLES_DIR / 'natural_fairness_paired_comparison.csv'
)

synthetic_fairness_metrics = pd.read_csv(
    TABLES_DIR / 'synthetic_fairness_metrics.csv'
)

synthetic_pooled_comparison = pd.read_csv(
    TABLES_DIR / 'synthetic_fairness_pooled_comparison.csv'
)

assert len(natural_fairness_ci) == 54
assert len(natural_paired_comparison) == 9
assert len(synthetic_fairness_metrics) == 1296
assert len(synthetic_pooled_comparison) == 36

# Verify all natural subgroup confusion counts

natural_groups_final = {
    'Any identity term': test_any_identity,
    **test_category_masks,
    **test_term_masks
}

for _, row in natural_identity_results.iterrows():

    rid = row['Run ID']
    outcome = row['Outcome']

    eligible = negative_masks[outcome]

    pred = audit_binary_predictions[rid]

    if outcome == 'any_label':
        flags = pred.any(axis=1)
    else:
        flags = pred[:, LABELS.index(outcome)]

    subgroup = (
        eligible &
        natural_groups_final[row['Identity Group']]
    )

    reference = eligible & no_identity_mask

    assert subgroup.sum() == row['Subgroup Negatives']
    assert reference.sum() == row['Reference Negatives']

    assert (
        subgroup & flags
    ).sum() == row['Subgroup FP']

    assert (
        reference & flags
    ).sum() == row['Reference FP']

print('All 324 natural subgroup records: Verified')

# Figure settings

MODEL_COLORS = {
    'Logistic Regression': '#D99A46',
    'DistilBERT': '#849DB5',
    'RoBERTa': '#326796'
}

model_names = [
    AUDIT_MODELS[rid]
    for rid in AUDIT_RUN_IDS
]

outcome_order = [
    'toxic', 'identity_hate', 'any_label'
]

outcome_names = [
    'Toxic', 'Identity hate', 'Any label'
]


def save_fairness_figure(fig, filename):

    fig.tight_layout()

    path = FIGURES_DIR / filename

    fig.savefig(
        path,
        dpi=300,
        bbox_inches='tight',
        facecolor='white'
    )

    with Image.open(path) as image:
        image.verify()

    plt.show()
    plt.close(fig)

    print('Figure saved:', filename)


# F03: Natural fairness gaps with confidence intervals

pooled_natural = natural_fairness_ci[
    natural_fairness_ci['Level'] == 'Overall'
]

fig, axes = plt.subplots(
    1, 3,
    figsize=(15, 4.5),
    sharey=True
)

for ax, outcome, title in zip(
    axes, outcome_order, outcome_names
):

    subset = pooled_natural[
        pooled_natural['Outcome'] == outcome
    ].set_index('Model').loc[model_names]

    for i, model_name in enumerate(model_names):

        row = subset.loc[model_name]

        point = row['FPR Gap (pp)']
        lower = row['Gap CI Lower (pp)']
        upper = row['Gap CI Upper (pp)']

        ax.errorbar(
            point,
            i,
            xerr=[
                [point - lower],
                [upper - point]
            ],
            fmt='o',
            markersize=7,
            capsize=4,
            color=MODEL_COLORS[model_name]
        )

    ax.axvline(
        0,
        color='black',
        linewidth=0.8,
        linestyle='--'
    )

    ax.set_title(title)
    ax.set_xlabel('FPR gap (percentage points)')
    ax.grid(axis='x', alpha=0.2)

axes[0].set_yticks(np.arange(3))
axes[0].set_yticklabels(model_names)
axes[0].invert_yaxis()

fig.suptitle(
    'Natural identity-term disparities with 95% confidence intervals',
    fontsize=13
)

save_fairness_figure(
    fig,
    'F03_natural_fairness_confidence_intervals.png'
)


# F04: Synthetic nonharmful prediction gaps

nonharmful_pooled = synthetic_pooled_comparison[
    synthetic_pooled_comparison['Frame Type'] == 'Nonharmful'
]

fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(outcome_order))
width = 0.24

for i, model_name in enumerate(model_names):

    subset = nonharmful_pooled[
        nonharmful_pooled['Model'] == model_name
    ].set_index('Outcome').loc[outcome_order]

    points = subset['Flag Rate Gap (pp)'].to_numpy()

    lower = subset['Gap CI Lower (pp)'].to_numpy()
    upper = subset['Gap CI Upper (pp)'].to_numpy()

    errors = np.maximum(
        np.vstack([
            points - lower,
            upper - points
        ]),
        0
    )

    ax.bar(
        x + (i - 1) * width,
        points,
        width=width,
        yerr=errors,
        capsize=3,
        color=MODEL_COLORS[model_name],
        label=model_name
    )

ax.axhline(0, color='black', linewidth=0.8)

ax.set_xticks(x)
ax.set_xticklabels(outcome_names)

ax.set_ylabel('Flag rate gap (percentage points)')

ax.set_title(
    'Counterfactual fairness on nonharmful sentences'
)

ax.legend()
ax.grid(axis='y', alpha=0.2)
ax.set_axisbelow(True)

save_fairness_figure(
    fig,
    'F04_synthetic_nonharmful_fairness.png'
)


# F05: Category-level synthetic toxicity sensitivity

category_synthetic = synthetic_fairness_metrics[
    (synthetic_fairness_metrics['Frame Type'] == 'Nonharmful') &
    (synthetic_fairness_metrics['Outcome'] == 'toxic') &
    (synthetic_fairness_metrics['Level'] == 'Category')
]

category_order = list(identity_lexicon)

gap_matrix = category_synthetic.pivot(
    index='Identity Group',
    columns='Model',
    values='Flag Rate Gap (pp)'
).loc[category_order, model_names]

score_matrix = category_synthetic.pivot(
    index='Identity Group',
    columns='Model',
    values='Mean Score Change'
).loc[category_order, model_names]

fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 6)
)

for ax, matrix, title, fmt, unit in [
    (
        axes[0],
        gap_matrix,
        'Toxic prediction gap',
        '.2f',
        'Percentage points'
    ),
    (
        axes[1],
        score_matrix,
        'Mean toxicity score change',
        '.3f',
        'Score difference'
    )
]:

    values = matrix.to_numpy()

    bound = max(
        float(np.abs(values).max()),
        1e-8
    )

    image = ax.imshow(
        values,
        cmap='RdBu_r',
        vmin=-bound,
        vmax=bound,
        aspect='auto'
    )

    ax.set_xticks(np.arange(3))

    ax.set_xticklabels(
        model_names,
        rotation=20,
        ha='right'
    )

    ax.set_yticks(
        np.arange(len(category_order))
    )

    ax.set_yticklabels(category_order)

    for r in range(values.shape[0]):

        for c in range(values.shape[1]):

            value = values[r, c]

            ax.text(
                c, r,
                format(value, fmt),
                ha='center',
                va='center',
                fontsize=10,
                color=(
                    'white'
                    if abs(value) > bound * 0.6
                    else 'black'
                )
            )

    ax.set_title(title)

    fig.colorbar(
        image,
        ax=ax,
        fraction=0.046,
        pad=0.04,
        label=unit
    )

fig.suptitle(
    'Identity-category sensitivity on matched nonharmful sentences'
)

save_fairness_figure(
    fig,
    'F05_synthetic_category_sensitivity.png'
)


# F06: Harmful sentence detection

harmful_pooled = synthetic_pooled_comparison[
    (synthetic_pooled_comparison['Frame Type'] == 'Harmful') &
    (synthetic_pooled_comparison['Outcome'] == 'toxic')
].set_index('Model').loc[model_names]

fig, ax = plt.subplots(figsize=(9, 5))

x = np.arange(len(model_names))
width = 0.34

bars1 = ax.bar(
    x - width / 2,
    harmful_pooled['Identity Flag Rate (%)'],
    width,
    label='Identity-containing sentences',
    color='#326796'
)

bars2 = ax.bar(
    x + width / 2,
    harmful_pooled['Control Flag Rate (%)'],
    width,
    label='Matched identity-free controls',
    color='#D99A46'
)

ax.bar_label(
    bars1,
    fmt='%.1f%%',
    padding=3
)

ax.bar_label(
    bars2,
    fmt='%.1f%%',
    padding=3
)

ax.set_xticks(x)
ax.set_xticklabels(model_names)

ax.set_ylim(0, 112)
ax.set_ylabel('Toxic detection rate (%)')

ax.set_title(
    'Detection of designed harmful counterfactual sentences'
)

ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.2)
ax.set_axisbelow(True)

save_fairness_figure(
    fig,
    'F06_synthetic_harmful_detection.png'
)


# Consolidate the major fairness findings

findings_rows = []

for rid in AUDIT_RUN_IDS:

    model_name = AUDIT_MODELS[rid]

    natural = pooled_natural[
        pooled_natural['Run ID'] == rid
    ].set_index('Outcome')

    nonharmful = nonharmful_pooled[
        (nonharmful_pooled['Run ID'] == rid) &
        (nonharmful_pooled['Outcome'] == 'toxic')
    ].iloc[0]

    orientation = category_synthetic[
        (category_synthetic['Run ID'] == rid) &
        (
            category_synthetic['Identity Group'] ==
            'Sexual orientation'
        )
    ].iloc[0]

    harmful = harmful_pooled.loc[model_name]

    findings_rows.append({
        'Run ID': rid,
        'Model': model_name,

        'Natural Toxic FPR Gap (pp)': (
            natural.loc['toxic', 'FPR Gap (pp)']
        ),

        'Natural Toxic Gap CI Lower (pp)': (
            natural.loc['toxic', 'Gap CI Lower (pp)']
        ),

        'Natural Toxic Gap CI Upper (pp)': (
            natural.loc['toxic', 'Gap CI Upper (pp)']
        ),

        'Natural Identity Hate FPR Gap (pp)': (
            natural.loc['identity_hate', 'FPR Gap (pp)']
        ),

        'Natural Any Label FPR Gap (pp)': (
            natural.loc['any_label', 'FPR Gap (pp)']
        ),

        'Synthetic Nonharmful Toxic FPR (%)': (
            nonharmful['Identity Flag Rate (%)']
        ),

        'Synthetic Control Toxic FPR (%)': (
            nonharmful['Control Flag Rate (%)']
        ),

        'Synthetic Nonharmful Upward Flips': int(
            nonharmful['Upward Prediction Flips']
        ),

        'Synthetic Sexual Orientation FPR (%)': (
            orientation['Identity Flag Rate (%)']
        ),

        'Synthetic Harmful Detection (%)': (
            harmful['Identity Flag Rate (%)']
        ),

        'Synthetic Harmful Control Detection (%)': (
            harmful['Control Flag Rate (%)']
        )
    })

fairness_key_findings = pd.DataFrame(findings_rows)

save_csv_atomic(
    TABLES_DIR / 'fairness_key_findings.csv',
    fairness_key_findings
)

print('\nFinal fairness findings')

display(fairness_key_findings.round(4))


# Verify all required result tables

required_tables = {
    'fairness_audit_model_roster.csv': 3,

    'identity_lexicon.csv': 30,
    'training_identity_associations.csv': 504,

    'natural_identity_subgroup_support.csv': 36,
    'natural_identity_fairness_metrics.csv': 324,
    'natural_identity_pooled_comparison.csv': 9,

    'synthetic_counterfactual_dataset.csv': 744,
    'synthetic_template_families.csv': 24,
    'synthetic_design_summary.csv': 6,

    'synthetic_inference_summary.csv': 3,

    'natural_fairness_confidence_intervals.csv': 54,
    'natural_fairness_paired_comparison.csv': 9,

    'synthetic_fairness_metrics.csv': 1296,
    'synthetic_fairness_pooled_comparison.csv': 36,

    'fairness_key_findings.csv': 3
}

artifact_inventory = []


def register_artifact(path):

    path = Path(path)

    assert path.is_file()
    assert path.stat().st_size > 0

    artifact_inventory.append({
        'path': str(path.relative_to(PROJECT_ROOT)),
        'size_bytes': int(path.stat().st_size),
        'sha256': calculate_sha256(path)
    })


for filename, expected_rows in required_tables.items():

    path = TABLES_DIR / filename

    frame = pd.read_csv(path)

    assert len(frame) == expected_rows, (
        filename, len(frame), expected_rows
    )

    register_artifact(path)

print('Required tables verified:', len(required_tables))


# Verify all required figures

required_figures = [
    'F01_training_identity_associations.png',
    'F02_natural_identity_fairness.png',
    'F03_natural_fairness_confidence_intervals.png',
    'F04_synthetic_nonharmful_fairness.png',
    'F05_synthetic_category_sensitivity.png',
    'F06_synthetic_harmful_detection.png'
]

for filename in required_figures:

    path = FIGURES_DIR / filename

    with Image.open(path) as image:

        assert image.format == 'PNG'

        assert image.width > 1000
        assert image.height > 1000

        image.verify()

    register_artifact(path)

print('Required figures verified:', len(required_figures))


# Register configurations and inference artifacts

required_configs = [
    'fairness_audit_setup.json',
    'identity_lexicon.json',
    'natural_identity_audit_config.json',
    'synthetic_counterfactual_config.json'
]

for filename in required_configs:

    path = RUN_CONFIGS_DIR / filename

    read_json(path)

    register_artifact(path)

for path in [
    prediction_path,
    PROJECT_ROOT / statistics_completion['matched_pair_path'],
    inference_completion_path,
    statistics_path
]:

    register_artifact(path)

assert len({
    record['path']
    for record in artifact_inventory
}) == len(artifact_inventory)


# Verify the complete frozen research protocol

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

assert calculate_sha256(
    lexicon_path
) == LEXICON_HASH

assert calculate_sha256(
    synthetic_path
) == SYNTHETIC_HASH

assert calculate_sha256(
    PROJECT_ROOT / synthetic_config['template_path']
) == synthetic_config['template_sha256']


# Record final fairness audit completion

fairness_completion = {
    'notebook': '09_fairness_audit',
    'status': 'Completed',

    'primary_run_id': PRIMARY_RUN_ID,
    'audit_run_ids': AUDIT_RUN_IDS,

    'identity_categories': 5,
    'identity_terms': 30,

    'official_test_samples': 63978,
    'natural_comment_groups': 63939,

    'natural_identity_comments': int(
        test_any_identity.sum()
    ),

    'natural_no_identity_comments': int(
        no_identity_mask.sum()
    ),

    'natural_metric_records': 324,
    'natural_ci_records': 54,
    'natural_paired_comparison_records': 9,

    'synthetic_comments': 744,
    'synthetic_template_families': 24,
    'synthetic_matched_pairs_per_model': 720,

    'synthetic_prediction_records': 2232,
    'synthetic_label_scores': 13392,
    'synthetic_fairness_records': 1296,

    'bootstrap_replicates': 1000,
    'confidence_level': 0.95,

    'verified_tables': len(required_tables),
    'verified_figures': len(required_figures),

    'artifact_inventory': artifact_inventory,

    'frozen_manifest_sha256': EXPECTED_FROZEN_HASH,
    'lexicon_sha256': LEXICON_HASH,
    'synthetic_dataset_sha256': SYNTHETIC_HASH,

    'synthetic_prediction_sha256': (
        inference_completion['prediction_sha256']
    ),

    'statistics_completion_sha256': (
        calculate_sha256(statistics_path)
    ),

    'validation_replay_verified': True,

    'model_training_performed': False,
    'thresholds_changed': False,
    'test_driven_selection_performed': False,

    'additional_inference_performed': True,
    'official_test_reinference_performed': False,

    'additional_inference_scope': (
        'Synthetic examples and validation replay'
    ),

    'independent_synthetic_human_validation_completed': False,

    'limitations': [
        'Identity mentions are not actual author demographics.',
        'Natural disparities are observational, not causal.',
        'Some identity terms have non-identity meanings.',
        'Synthetic labels represent designed intent.',
        'Synthetic frames are limited in diversity.',
        'Some identity terms have low natural test support.',
        'Bootstrap intervals are conditional on fixed models.',
        'Confidence intervals are not multiplicity adjusted.',
        'Model comparisons also differ in training sample size.',
        'Reduced disparity does not establish absence of bias.'
    ],

    'next_notebook': '10_explainability',

    'completed_at': datetime.now(
        timezone.utc
    ).isoformat()
}

completion_path = (
    LOGS_DIR / 'fairness_audit_completion_summary.json'
)

save_json_atomic(
    completion_path,
    fairness_completion
)

# Verify saved inventory fingerprints

for record in artifact_inventory:

    verify_file(record)

print('\nNotebook 09 completion summary')

for key, value in fairness_completion.items():

    if key not in [
        'artifact_inventory',
        'limitations'
    ]:

        print(f'{key}: {value}')

print('\nVerified artifact inventory:', len(artifact_inventory))
print('Frozen protocol: Unchanged')
print('Additional training: None')
print('Cell 7 completed successfully.')
print('Notebook 09 completed successfully.')